# Notebook 09 — Benchmark: MoE vs. Conventional Forecasting Models

---

This is the last notebook of the MoEForecaster pipeline (Mixture-of-Experts forecaster of hourly electricity demand for Colombia's National Interconnected System, SIN):

`01 data acquisition → 02 preprocessing / feature engineering → 03 clustering → 04 cluster analysis + classifiers → 05 expert selection → 06 gating design → 07 HPO → 08 production assembly → ` **`09 benchmark`**

It compares the production MoE model assembled in notebook 08 against conventional baselines from three model families (statistical, machine learning and deep learning), using the same feature table, the same test period and the same error metrics.

## Objective

Benchmark the proposed MoE forecaster against 30 conventional models (3 families × 10 models) and analyse the comparison along four dimensions:

1. **Predictive performance**: MAE, RMSE, MAPE and R² on a held-out test set, plus statistical significance tests on the per-observation errors.
2. **Computational efficiency**: training time, inference time and training throughput.
3. **Interpretability**: a 1–5 qualitative interpretability scale, SHAP feature attributions for the baselines, and an analysis of the MoE routing (t-SNE/UMAP projections, expert activation by context, routing stability).
4. **Robustness**: an out-of-distribution stress test on the COVID-19 period (2020–2021), which was excluded from the training data of every model.

The notebook ends with an out-of-sample forecast of January 2026 that is compared with the actual demand downloaded from the XM S.A. API.

**Workflow.** All 30 baselines are first *screened* with default hyperparameters (fit on the training split, scored on the validation split). The models retained for the final comparison are then tuned by random-search hyperparameter optimization (HPO) and evaluated on the test split together with the MoE.

## Inputs

| File | Produced by | Content |
|---|---|---|
| `DATA_DIR/processed/features.parquet` | Notebook 02 | Hourly demand and engineered features (calendar, holiday and demand-lag features) |
| `MODELS_DIR/production/moe_forecaster.pkl` | Notebook 08 | Serialized production `MoEForecaster` (experts + cluster classifiers + gating) |
| XM S.A. API (`servapibi.xm.com.co`, metric `DemaReal`) | Downloaded in Section 11 | Actual hourly SIN demand for January 2026 (requires internet access) |

## Outputs

All outputs are written to `RESULTS_DIR/benchmark/` (`BENCH_DIR` in the code).

**Tables and serialized intermediate results**

| File | Content |
|---|---|
| `stat_screening_df.csv`, `ml_screening_df.csv`, `dl_screening_df.csv` | Validation-set screening results of each family |
| `stat_results.pkl`, `ml_results.pkl`, `dl_results.pkl` | Screening results (fitted models, metrics, times) |
| `top3_stat.pkl`, `top3_ml.pkl`, `top3_dl.pkl` | Models selected for HPO in each family |
| `hpo_results.pkl`, `dl_hpo_results.pkl` | Tuned models, best hyperparameters and test metrics |
| `benchmark_summary.csv` | Master table: test-set metrics and times of the MoE and the tuned baselines |
| `scalability_analysis.csv` | Training throughput (samples/s) |
| `interpretability_scores.csv` | Interpretability score (1–5) per model |
| `routing_stability.json` | k-NN stability of the MoE routing |
| `tukey_hsd_results.csv`, `pairwise_wilcoxon_pvalues.csv` | Post-hoc statistical tests |
| `covid_stress_test_results.csv`, `covid_stress_conclusions.json` | COVID-19 stress test |
| `metrics_jan_2026.csv`, `metrics_jan_2026_one_week.csv`, `metrics_jan_2026_two_week.csv` | Error metrics of the January 2026 forecast |
| `benchmark_results.json` | Final summary (master table, ANOVA result, best model) |

**Figures** (`.png`): `fig_predictive_performance`, `fig_radar_comparison`, `fig_computational_efficiency_train`, `fig_computational_efficiency_inf`, `fig_interpretability_vs_performance`, `fig_shap_summary_comparison`, `fig_shap_beeswarm_best`, `fig_shap_ranking_correlation`, `fig_routing_tsne_umap`, `fig_expert_activation_analysis`, `fig_routing_stability`, `fig_tukey_hsd`, `fig_pairwise_wilcoxon`, `fig_error_distributions`, `fig_covid_timeseries`, `fig_covid_degradation_heatmap`, `fig_covid_error_boxplots`, `fig_covid_holy_week_zoom`, `demand_forecast_january_2026`, `demand_forecast_january_2026_one_week`, `demand_forecast_january_2026_two_week`.

## Environment setup

This notebook runs unchanged on a local machine (pip/venv or conda), in Docker and in Google Colab.

**`OUTPUT_ROOT`** (next cell) is the folder where the pipeline stores `data/`, `models/` and `results/`:

| Value | Where artifacts are stored |
|---|---|
| `None` (default) | The repository folder, or the folder in the `MOE_OUTPUT_ROOT` environment variable if it is set |
| `"/content/drive/MyDrive/MoEForecaster"` | Google Drive (Colab only; Drive is mounted automatically) |
| `"/any/absolute/path"` or `"relative/path"` | Any folder; relative paths are resolved from the repository root |

Use the **same `OUTPUT_ROOT` in every notebook**: each notebook reads the artifacts written by the previous ones. Setting the `MOE_OUTPUT_ROOT` environment variable once is the easiest way to keep it consistent.

The cell after it finds the repository, which it clones first when running in Colab. In Colab it also installs the pinned dependencies from `requirements.txt` and makes the `moe_forecaster` package in `src/` importable.

In [ ]:
# ============================================================
# PARAMETERS: the only cell you may need to edit
# ============================================================
# Folder for data/, models/ and results/ (see the table above).
# None -> repository folder, or $MOE_OUTPUT_ROOT if that variable is set.
OUTPUT_ROOT = None

In [ ]:
# ============================================================
# ENVIRONMENT BOOTSTRAP: no edits needed
# ============================================================
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/astorress/MoEForecaster.git"
IN_COLAB = "google.colab" in sys.modules


def find_repo_root(start):
    """Walk up from `start` until the folder that contains src/moe_forecaster/."""
    for folder in [start, *start.parents]:
        if (folder / "src" / "moe_forecaster").is_dir():
            return folder
    return None


REPO_ROOT = find_repo_root(Path.cwd().resolve())

if REPO_ROOT is None and IN_COLAB:
    # Colab opens the notebook on its own, without the rest of the repository
    REPO_ROOT = Path("/content/MoEForecaster")
    if not REPO_ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_ROOT)], check=True)

if REPO_ROOT is None:
    raise RuntimeError(
        "Repository root not found. Start Jupyter from inside the MoEForecaster folder "
        "(local, conda or Docker), or open the notebook in Google Colab."
    )

if IN_COLAB:
    # Install the pinned versions once per Colab runtime. Colab's preinstalled
    # PyTorch (built with CUDA) is kept, so the torch pin is skipped here.
    marker = Path("/content/.moe_requirements_installed")
    if not marker.exists():
        reqs = [
            line for line in (REPO_ROOT / "requirements.txt").read_text().splitlines()
            if line.strip() and not line.startswith(("#", "torch"))
        ]
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *reqs], check=True)
        marker.touch()
        print("Dependencies installed. If Colab asks to restart the session, restart it and run the notebook again.")

if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from moe_forecaster.config import setup_paths

DATA_DIR, MODELS_DIR, RESULTS_DIR = setup_paths(OUTPUT_ROOT, repo_root=REPO_ROOT)

## 1. Configuration and imports

Global settings of the benchmark: random seed (fixed for NumPy and PyTorch), number of parallel jobs, random-search budget and number of time-series cross-validation folds. The budget of $n = 60$ random-search iterations follows Bergstra & Bengio (2012) (see Section 8).

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
PROD_DIR    = f"{MODELS_DIR}/production"
RANDOM_SEED = 42
N_JOBS      = -1

# Bergstra & Bengio (2012): n=60 → P(top 5%) ≥ 0.95
N_RANDOM_SEARCH_ITER = 60
N_CV_FOLDS = 3  # TimeSeriesSplit folds

import os, json, pickle, time, warnings, gc, copy
from datetime import datetime, timedelta
from pathlib import Path
from collections import OrderedDict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from scipy import stats as sp_stats
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                             mean_absolute_percentage_error, r2_score)
from sklearn.experimental import enable_halving_search_cv  # required to import HalvingRandomSearchCV
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV, HalvingRandomSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline

# ML models
from sklearn.linear_model import (Ridge, Lasso, ElasticNet, BayesianRidge,
                                  HuberRegressor, SGDRegressor)
from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor,
                              ExtraTreesRegressor, AdaBoostRegressor,
                              BaggingRegressor)
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
import lightgbm as lgb
import xgboost as xgb

# Statistical models
import statsmodels.api as sm
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.ar_model import AutoReg

# Deep Learning
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings('ignore')
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

plt.rcParams.update({
    'figure.dpi': 150, 'savefig.dpi': 300,
    'font.family': 'serif', 'font.size': 11,
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'legend.fontsize': 9, 'axes.grid': True, 'grid.alpha': 0.3,
})

# Output folder of the benchmark (created by the environment setup cell)
BENCH_DIR = Path(RESULTS_DIR) / 'benchmark'
print(f"Benchmark output folder: {BENCH_DIR}")

---
## 2. Data loading and preparation

All models are built from the same `features.parquet` file (notebook 02) that feeds the MoE pipeline, so that every model sees the same observations.

**Feature set.** Every baseline uses the same reduced feature set (`calendar_lags`):
- **Calendar features**: cyclical encodings of hour, day of week and month (`*_sin`, `*_cos`), `is_holiday`, `is_bridge` (bridge holiday), `days_to_next_holiday` and `days_from_prev_holiday`.
- **Demand lags**: every `demand_lag_*` column of the feature table.

The broader feature selection (all numeric columns except the target and the cluster labels) is kept in the code as a commented-out alternative and is not used. Rows with missing values in the target or in the selected features are dropped.

In [ ]:
# ============================================================
# DATA LOADING
# ============================================================
features_path = Path(DATA_DIR) / 'processed' / 'features.parquet'

if features_path.exists():
    df = pd.read_parquet(features_path)
    if not isinstance(df.index, pd.DatetimeIndex):
        for col in ['timestamp', 'datetime', 'date']:
            if col in df.columns:
                df = df.set_index(col)
                break
    df = df.sort_index()
    print(f"✓ Data loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"  Range: {df.index.min()} → {df.index.max()}")
else:
    # No fallback to synthetic data: the benchmark must run on the real data
    raise FileNotFoundError(
        f"{features_path} not found. Run notebooks 01 and 02 first (make pipeline)."
    )

# Target and features
calendar_only = [
    'hour_sin', 'hour_cos', 'dayofweek_sin', 'dayofweek_cos', 'month_sin', 'month_cos',
    'is_holiday', 'is_bridge', 'days_to_next_holiday', 'days_from_prev_holiday',
]

all_cols = list(df.columns)

# Demand-lag features
lag_features = sorted([c for c in all_cols if c.startswith('demand_lag_')])

calendar_lags = calendar_only + lag_features

TARGET = 'demand'
# Alternative (not used): all numeric columns except the target and the cluster labels
EXCLUDE_COLS = [TARGET, 'type_day', 'type_week', 'cluster']
#feature_cols = [c for c in df.columns if c not in EXCLUDE_COLS
#                and not c.startswith('type_') and df[c].dtype in ['float64','float32','int64','int32']]

feature_cols = calendar_lags

print(f"\n  Target: {TARGET}")
print(f"  Features ({len(feature_cols)}): {feature_cols[:10]}...")

# Drop rows with missing values
df_clean = df[[TARGET] + feature_cols].dropna()
print(f"  Clean rows: {df_clean.shape[0]:,}")

In [ ]:
df_clean.head()

The years 2020 and 2021 are removed from the benchmark data because the COVID-19 pandemic and the 2021 national strike distorted the demand patterns. The same two years are excluded in the MoE pipeline. They are used again in Section 10.5 as an out-of-distribution stress test.

In [ ]:
df_clean = df_clean[~df_clean.index.year.isin([2020, 2021])]

In [ ]:
df_clean.head()

In [ ]:
df_clean.shape

### 2.1 Chronological train / validation / test split

The cleaned series is split chronologically, without shuffling, so that every model is evaluated on data that comes after the data it was fitted on:
- **Train**: first 70% of the hourly observations.
- **Validation**: next 15%. Used to screen the 30 baselines with default hyperparameters (Sections 4–6) and as the early-stopping set of the deep-learning models.
- **Test**: last 15%. Used for the final comparison with the MoE (Section 9).

For HPO, train and validation are concatenated (`X_trainval`) and searched with `TimeSeriesSplit` cross-validation (Section 8). The standardization scalers used by the deep-learning models are fitted on the training split only.

In [ ]:
# ============================================================
# CHRONOLOGICAL SPLIT
# ============================================================
n_total = len(df_clean)
n_train = int(n_total * 0.70)
n_val   = int(n_total * 0.15)
n_test  = n_total - n_train - n_val

df_train = df_clean.iloc[:n_train]
df_val   = df_clean.iloc[n_train:n_train + n_val]
df_test  = df_clean.iloc[n_train + n_val:]

X_train, y_train = df_train[feature_cols], df_train[TARGET]
X_val,   y_val   = df_val[feature_cols],   df_val[TARGET]
X_test,  y_test  = df_test[feature_cols],  df_test[TARGET]

# Train + validation combined, for HPO with TimeSeriesSplit
X_trainval = pd.concat([X_train, X_val])
y_trainval = pd.concat([y_train, y_val])

# Target series alone, for the univariate time-series models
ts_train = y_train.copy()
ts_val   = y_val.copy()
ts_test  = y_test.copy()

# Scalers for the deep-learning models (fitted on the training split)
scaler_X = StandardScaler().fit(X_train)
scaler_y = StandardScaler().fit(y_train.values.reshape(-1, 1))

print(f"Train: {len(df_train):,} ({df_train.index.min().date()} → {df_train.index.max().date()})")
print(f"Val:   {len(df_val):,}  ({df_val.index.min().date()} → {df_val.index.max().date()})")
print(f"Test:  {len(df_test):,}  ({df_test.index.min().date()} → {df_test.index.max().date()})")

### 2.2 Evaluation helpers

- `compute_metrics`: MAE, RMSE, MAPE (%) and R². MAPE is computed only over observations with non-zero actual demand.
- `timer_context`: wall-clock timer (`time.perf_counter`) used to measure training and inference times.
- `print_screening_results`: summary table of a family screening, sorted by validation MAPE.

In [ ]:
# ============================================================
# METRICS AND UTILITIES
# ============================================================
def compute_metrics(y_true, y_pred, prefix=''):
    """Compute standard regression metrics (MAE, RMSE, MAPE in %, R²)."""
    y_true = np.asarray(y_true).flatten()
    y_pred = np.asarray(y_pred).flatten()
    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    return {
        f'{prefix}MAE':  mean_absolute_error(y_true, y_pred),
        f'{prefix}RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        f'{prefix}MAPE': mape,
        f'{prefix}R2':   r2_score(y_true, y_pred),
    }

def timer_context():
    """Context manager that measures elapsed wall-clock time."""
    class Timer:
        def __enter__(self):
            self.start = time.perf_counter()
            return self
        def __exit__(self, *args):
            self.elapsed = time.perf_counter() - self.start
    return Timer()

def print_screening_results(results_dict, family_name):
    """Print and return the screening summary table of one model family."""
    rows = []
    for name, r in results_dict.items():
        rows.append({
            'Model': name,
            'MAPE_val (%)': r['val_metrics']['MAPE'],
            'RMSE_val': r['val_metrics']['RMSE'],
            'MAE_val': r['val_metrics']['MAE'],
            'R2_val': r['val_metrics']['R2'],
            'Train_time (s)': r['train_time'],
        })
    res_df = pd.DataFrame(rows).sort_values('MAPE_val (%)')
    print(f"\n{'='*80}")
    print(f"  SCREENING — {family_name} family ({len(res_df)} models)")
    print(f"{'='*80}")
    print(res_df.to_string(index=False, float_format='{:.4f}'.format))
    return res_df

print("✓ Helper functions defined")

---
## 3. Proposed model: loading the MoE

The production MoE serialized by notebook 08 (`MODELS_DIR/production/moe_forecaster.pkl`) is loaded unchanged and used to predict the test split defined in Section 2.1. Its test predictions are the reference against which all baselines are compared. The `MoEForecaster` class is imported from `src/moe_forecaster/forecaster.py` so that the pickle can be deserialized.

In [ ]:
# ============================================================
# IMPORT THE MoEForecaster CLASS
# ============================================================
# The class lives in src/moe_forecaster/forecaster.py so that moe_forecaster.pkl
# can be loaded outside the notebook that created it: a class defined inside a
# notebook is pickled as __main__.MoEForecaster and only exists in that session.
from moe_forecaster import MoEForecaster
print('MoEForecaster class imported from src/moe_forecaster.')

In [ ]:
# ============================================================
# LOAD THE MoE MODEL AND PREDICT THE TEST SET
# ============================================================
moe_path = Path(PROD_DIR) / 'moe_forecaster.pkl'

if moe_path.exists():
    try:
        with open(moe_path, 'rb') as f:
            moe_model = pickle.load(f)
        print("✓ MoE model loaded from production folder")

        # Inspect the loaded object
        print(f"\nLoaded object type: {type(moe_model)}")
        print("Model attributes (examples):")
        for attr in ['num_experts', 'input_dim', 'experts', 'gating_network', 'scalers']:
            if hasattr(moe_model, attr):
                print(f"  - {attr}: {getattr(moe_model, attr)}")

        # MoE predictions on the test set
        with timer_context() as t_moe_infer:
            moe_preds_test = moe_model.predict(df_test)

        moe_metrics_test = compute_metrics(y_test, moe_preds_test)
        moe_train_time = 0  # Already trained; training time is documented separately
        moe_infer_time = t_moe_infer.elapsed

        print(f"\n  MoE Test — MAPE: {moe_metrics_test['MAPE']:.4f}% | RMSE: {moe_metrics_test['RMSE']:.2f}")
        print(f"  Test inference time: {moe_infer_time:.3f}s")
    except Exception as e:
        # The benchmark is meaningless without the main model: stop instead of continuing with NaN
        raise RuntimeError(f"Error while loading or evaluating the MoE model: {e}") from e
else:
    # No synthetic fallback: if an artifact from an earlier notebook is missing,
    # that notebook must be run; the benchmark does not continue with made-up data.
    raise FileNotFoundError(f"{moe_path} not found. Run notebook 08 first (make pipeline).")

---
## 4. Statistical family — 10 models (screening)

Ten classical statistical and time-series models. Models that accept exogenous inputs use the feature matrix of Section 2; purely univariate models use only the `demand` series. In this screening stage every model is fitted on the training split with default or fixed hyperparameters and scored on the validation split.

### Models
1. **OLS** (linear regression on the features)
2. **Ridge** (L2 regularization)
3. **Lasso** (L1 regularization)
4. **ElasticNet** (L1 + L2)
5. **BayesianRidge**
6. **HuberRegressor** (robust to outliers)
7. **AutoReg** (pure autoregressive model, statsmodels, 168 lags), forecasting the whole validation horizon from the end of the training series
8. **Exponential Smoothing** (additive Holt-Winters, weekly seasonality, fitted on daily means)
9. **SARIMAX** $(2,1,2)\times(1,1,1)_7$ without exogenous inputs, fitted on daily means
10. **Prophet** (daily, weekly and yearly seasonality, hourly data)

> **Note**: models 1–6 are linear models on the engineered features; models 7–10 are classical univariate time-series models. Holt-Winters and SARIMAX are fitted on daily means to keep them computationally feasible; their daily forecasts are linearly interpolated to hourly resolution so they are scored against the same hourly validation targets. If a time-series model fails to fit, it receives a penalty score (MAPE = 100%) so that it is not selected.

In [ ]:
# ============================================================
# 4. STATISTICAL MODELS — SCREENING
# ============================================================
from sklearn.linear_model import LinearRegression

stat_results = OrderedDict()

# --- 4.1  OLS ---
print("Training: OLS (Linear Regression)...")
with timer_context() as t:
    m = LinearRegression(n_jobs=N_JOBS).fit(X_train, y_train)
stat_results['OLS'] = {
    'model': m,
    'val_metrics': compute_metrics(y_val, m.predict(X_val)),
    'train_time': t.elapsed,
    'type': 'feature_based'
}

# --- 4.2  Ridge ---
print("Training: Ridge...")
with timer_context() as t:
    m = Ridge(alpha=1.0, random_state=RANDOM_SEED).fit(X_train, y_train)
stat_results['Ridge'] = {
    'model': m,
    'val_metrics': compute_metrics(y_val, m.predict(X_val)),
    'train_time': t.elapsed,
    'type': 'feature_based'
}

# --- 4.3  Lasso ---
print("Training: Lasso...")
with timer_context() as t:
    m = Lasso(alpha=1.0, random_state=RANDOM_SEED, max_iter=5000).fit(X_train, y_train)
stat_results['Lasso'] = {
    'model': m,
    'val_metrics': compute_metrics(y_val, m.predict(X_val)),
    'train_time': t.elapsed,
    'type': 'feature_based'
}

# --- 4.4  ElasticNet ---
print("Training: ElasticNet...")
with timer_context() as t:
    m = ElasticNet(alpha=1.0, l1_ratio=0.5, random_state=RANDOM_SEED, max_iter=5000).fit(X_train, y_train)
stat_results['ElasticNet'] = {
    'model': m,
    'val_metrics': compute_metrics(y_val, m.predict(X_val)),
    'train_time': t.elapsed,
    'type': 'feature_based'
}

# --- 4.5  BayesianRidge ---
print("Training: BayesianRidge...")
with timer_context() as t:
    m = BayesianRidge(max_iter=500).fit(X_train, y_train)
stat_results['BayesianRidge'] = {
    'model': m,
    'val_metrics': compute_metrics(y_val, m.predict(X_val)),
    'train_time': t.elapsed,
    'type': 'feature_based'
}

# --- 4.6  HuberRegressor ---
print("Training: HuberRegressor...")
with timer_context() as t:
    m = HuberRegressor(max_iter=500).fit(X_train, y_train)
stat_results['HuberRegressor'] = {
    'model': m,
    'val_metrics': compute_metrics(y_val, m.predict(X_val)),
    'train_time': t.elapsed,
    'type': 'feature_based'
}

# --- 4.7  AutoReg (statsmodels) ---
print("Training: AutoReg (lags=168)...")
try:
    with timer_context() as t:
        ar_model = AutoReg(ts_train.values, lags=168, old_names=False).fit()
    ar_preds_val = ar_model.predict(start=len(ts_train), end=len(ts_train)+len(ts_val)-1)
    stat_results['AutoReg'] = {
        'model': ar_model,
        'val_metrics': compute_metrics(y_val.values, ar_preds_val),
        'train_time': t.elapsed,
        'type': 'ts_univariate'
    }
except Exception as e:
    print(f"  ⚠️ AutoReg failed: {e}")
    stat_results['AutoReg'] = {
        'model': None,
        'val_metrics': {'MAE': 1e9, 'RMSE': 1e9, 'MAPE': 100, 'R2': -1},
        'train_time': 0, 'type': 'ts_univariate'
    }

# --- 4.8  Exponential Smoothing (Holt-Winters) ---
print("Training: Holt-Winters Exponential Smoothing...")
try:
    # Daily resampling to keep the fit computationally feasible
    ts_daily_train = ts_train.resample('D').mean()
    ts_daily_val   = ts_val.resample('D').mean()
    with timer_context() as t:
        hw_model = ExponentialSmoothing(
            ts_daily_train, trend='add', seasonal='add',
            seasonal_periods=7
        ).fit(optimized=True)
    hw_preds_daily = hw_model.forecast(len(ts_daily_val))
    # Interpolate back to hourly resolution so that it is scored on the same hourly targets
    hw_preds_hourly = np.interp(
        np.linspace(0, len(hw_preds_daily)-1, len(ts_val)),
        np.arange(len(hw_preds_daily)),
        hw_preds_daily.values
    )
    stat_results['HoltWinters'] = {
        'model': hw_model,
        'val_metrics': compute_metrics(y_val.values, hw_preds_hourly),
        'train_time': t.elapsed,
        'type': 'ts_univariate'
    }
except Exception as e:
    print(f"  ⚠️ HoltWinters failed: {e}")
    stat_results['HoltWinters'] = {
        'model': None,
        'val_metrics': {'MAE': 1e9, 'RMSE': 1e9, 'MAPE': 100, 'R2': -1},
        'train_time': 0, 'type': 'ts_univariate'
    }

# --- 4.9  SARIMAX ---
print("Training: SARIMAX (daily resampling)...")
try:
    ts_daily_train = ts_train.resample('D').mean()
    ts_daily_val   = ts_val.resample('D').mean()
    with timer_context() as t:
        sarimax_model = SARIMAX(
            ts_daily_train, order=(2,1,2),
            seasonal_order=(1,1,1,7),
            enforce_stationarity=False, enforce_invertibility=False
        ).fit(disp=False, maxiter=200)
    sar_preds_daily = sarimax_model.forecast(len(ts_daily_val))
    sar_preds_hourly = np.interp(
        np.linspace(0, len(sar_preds_daily)-1, len(ts_val)),
        np.arange(len(sar_preds_daily)),
        sar_preds_daily.values
    )
    stat_results['SARIMAX'] = {
        'model': sarimax_model,
        'val_metrics': compute_metrics(y_val.values, sar_preds_hourly),
        'train_time': t.elapsed,
        'type': 'ts_univariate'
    }
except Exception as e:
    print(f"  ⚠️ SARIMAX failed: {e}")
    stat_results['SARIMAX'] = {
        'model': None,
        'val_metrics': {'MAE': 1e9, 'RMSE': 1e9, 'MAPE': 100, 'R2': -1},
        'train_time': 0, 'type': 'ts_univariate'
    }

# --- 4.10  Prophet ---
print("Training: Prophet...")
try:
    from prophet import Prophet
    prophet_train = pd.DataFrame({
        'ds': ts_train.index, 'y': ts_train.values
    })
    with timer_context() as t:
        prophet_model = Prophet(
            daily_seasonality=True, weekly_seasonality=True,
            yearly_seasonality=True, changepoint_prior_scale=0.05
        )
        prophet_model.fit(prophet_train)
    future = prophet_model.make_future_dataframe(periods=len(ts_val), freq='h')
    prophet_preds_all = prophet_model.predict(future)
    prophet_preds_val = prophet_preds_all['yhat'].iloc[-len(ts_val):].values
    stat_results['Prophet'] = {
        'model': prophet_model,
        'val_metrics': compute_metrics(y_val.values, prophet_preds_val),
        'train_time': t.elapsed,
        'type': 'ts_univariate'
    }
except Exception as e:
    print(f"  ⚠️ Prophet failed: {e}")
    stat_results['Prophet'] = {
        'model': None,
        'val_metrics': {'MAE': 1e9, 'RMSE': 1e9, 'MAPE': 100, 'R2': -1},
        'train_time': 0, 'type': 'ts_univariate'
    }

# --- Results ---
stat_screening_df = print_screening_results(stat_results, 'Statistical')

---
## 5. Machine-learning family — 10 models (screening)

All machine-learning models use the same feature matrix (`X_train`, `X_val`). Unlike the MoE, they are single global models: the data are not segmented by day type or week type. Hyperparameters are mostly library defaults with a fixed number of estimators and a fixed random seed.

### Models
1. Random Forest
2. Gradient Boosting (scikit-learn)
3. XGBoost
4. LightGBM
5. Extra Trees
6. AdaBoost
7. SVR (RBF kernel, with feature standardization)
8. KNN regressor (k = 10, distance weighting)
9. Bagging regressor
10. Decision tree (simple baseline)

In [ ]:
# ============================================================
# 5. MACHINE-LEARNING MODELS — SCREENING
# ============================================================
ml_results = OrderedDict()

ml_models = {
    'RandomForest': RandomForestRegressor(
        n_estimators=100, random_state=RANDOM_SEED, n_jobs=N_JOBS),
    'GradientBoosting': GradientBoostingRegressor(
        n_estimators=100, random_state=RANDOM_SEED),
    'XGBoost': xgb.XGBRegressor(
        n_estimators=100, random_state=RANDOM_SEED, n_jobs=N_JOBS, verbosity=0),
    'LightGBM': lgb.LGBMRegressor(
        n_estimators=100, random_state=RANDOM_SEED, n_jobs=N_JOBS, verbose=-1),
    'ExtraTrees': ExtraTreesRegressor(
        n_estimators=100, random_state=RANDOM_SEED, n_jobs=N_JOBS),
    'AdaBoost': AdaBoostRegressor(
        n_estimators=50, random_state=RANDOM_SEED),
    'SVR': Pipeline([
        ('scaler', StandardScaler()),
        ('svr', SVR(kernel='rbf'))
    ]),
    'KNN': KNeighborsRegressor(n_neighbors=10, n_jobs=N_JOBS, weights='distance'),
    'Bagging': BaggingRegressor(
        n_estimators=50, random_state=RANDOM_SEED, n_jobs=N_JOBS),
    'DecisionTree': DecisionTreeRegressor(random_state=RANDOM_SEED),
}

for name, model in ml_models.items():
    print(f"Training: {name}...")
    with timer_context() as t:
        model.fit(X_train, y_train)
    preds_val = model.predict(X_val)
    ml_results[name] = {
        'model': model,
        'val_metrics': compute_metrics(y_val, preds_val),
        'train_time': t.elapsed,
        'type': 'feature_based'
    }

ml_screening_df = print_screening_results(ml_results, 'Machine Learning')

---
## 6. Deep-learning family — 10 models (PyTorch)

Ten deep-learning architectures for time-series forecasting. All of them share:
- The same preprocessing: features and target standardized with `StandardScaler` fitted on the training split.
- An input window of 168 hourly steps (one week) of the feature vectors.
- A one-step-ahead output: the (standardized) demand at the hour that follows the window.
- Training with Adam (weight decay $10^{-5}$), MSE loss, gradient clipping (norm 1.0), `ReduceLROnPlateau` and early stopping on the validation loss (patience 7, at most 50 epochs, batch size 32). The weights with the lowest validation loss are restored.

Because each prediction needs the previous 168 hours, the first 168 observations of every evaluation split have no prediction; the deep-learning metrics are computed on the remaining observations.

### Models
1. **LSTM** — Long Short-Term Memory
2. **GRU** — Gated Recurrent Unit
3. **BiLSTM** — Bidirectional LSTM
4. **Stacked LSTM** — three stacked LSTM layers
5. **CNN-1D** — 1-D convolutional network
6. **TCN** — Temporal Convolutional Network
7. **Transformer** — encoder-only Transformer
8. **CNN-LSTM** — hybrid CNN + LSTM
9. **Attention-LSTM** — LSTM with an attention mechanism
10. **MLP** — multi-layer perceptron on the flattened window (deep-learning baseline)

In [ ]:
# ============================================================
# 6.0  DATASET AND DEEP-LEARNING UTILITIES
# ============================================================
WINDOW_SIZE = 168  # 1 week
DL_EPOCHS = 50
DL_BATCH = 32
DL_LR = 1e-3
DL_PATIENCE = 7
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class TimeSeriesDataset(Dataset):
    """Sequence dataset: sliding window over the features; the target is the step after the window."""
    def __init__(self, X, y, window_size=WINDOW_SIZE):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        self.window_size = window_size

    def __len__(self):
        return len(self.X) - self.window_size

    def __getitem__(self, idx):
        x_seq = self.X[idx:idx+self.window_size]
        y_target = self.y[idx+self.window_size]
        return x_seq, y_target

def prepare_dl_data(X_df, y_series, scaler_X, scaler_y, fit=False):
    """Scale features and target for the DataLoader."""
    if fit:
        X_scaled = scaler_X.fit_transform(X_df)
        y_scaled = scaler_y.fit_transform(y_series.values.reshape(-1,1)).flatten()
    else:
        X_scaled = scaler_X.transform(X_df)
        y_scaled = scaler_y.transform(y_series.values.reshape(-1,1)).flatten()
    return X_scaled, y_scaled

def train_dl_model(model, train_loader, val_loader, epochs=DL_EPOCHS,
                   lr=DL_LR, patience=DL_PATIENCE):
    """Generic training loop with early stopping on the validation loss."""
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
    criterion = nn.MSELoss()
    best_val_loss = float('inf')
    best_state = None
    patience_counter = 0
    history = {'train_loss': [], 'val_loss': []}

    for epoch in range(epochs):
        # Train
        model.train()
        train_losses = []
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            pred = model(xb).squeeze()
            loss = criterion(pred, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_losses.append(loss.item())

        # Validation
        model.eval()
        val_losses = []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                pred = model(xb).squeeze()
                val_losses.append(criterion(pred, yb).item())

        train_loss = np.mean(train_losses)
        val_loss = np.mean(val_losses)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    model.load_state_dict(best_state)
    return model, history

def predict_dl(model, data_loader, scaler_y):
    """Generate predictions and transform them back to the original scale."""
    model.eval()
    preds = []
    with torch.no_grad():
        for xb, _ in data_loader:
            xb = xb.to(device)
            out = model(xb).squeeze().cpu().numpy()
            preds.append(out)
    preds = np.concatenate(preds)
    return scaler_y.inverse_transform(preds.reshape(-1,1)).flatten()

# Prepare data
X_train_sc, y_train_sc = prepare_dl_data(X_train, y_train, scaler_X, scaler_y, fit=True)
X_val_sc,   y_val_sc   = prepare_dl_data(X_val,   y_val,   scaler_X, scaler_y)

train_ds = TimeSeriesDataset(X_train_sc, y_train_sc, WINDOW_SIZE)
val_ds   = TimeSeriesDataset(X_val_sc,   y_val_sc,   WINDOW_SIZE)

train_loader = DataLoader(train_ds, batch_size=DL_BATCH, shuffle=False, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=DL_BATCH, shuffle=False)

n_features = X_train.shape[1]
print(f"✓ Deep-learning data prepared: {len(train_ds)} train / {len(val_ds)} val samples")
print(f"  Features: {n_features} | Window: {WINDOW_SIZE} | Device: {device}")

In [ ]:
# ============================================================
# 6.1  DEEP-LEARNING ARCHITECTURES
# ============================================================

class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers,
                            batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_dim, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

class GRUModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, num_layers,
                          batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_dim, 1)
    def forward(self, x):
        out, _ = self.gru(x)
        return self.fc(out[:, -1, :])

class BiLSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers,
                            batch_first=True, dropout=dropout, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

class StackedLSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, dropout=0.2):
        super().__init__()
        self.lstm1 = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.drop1 = nn.Dropout(dropout)
        self.lstm2 = nn.LSTM(hidden_dim, hidden_dim * 3 // 4, batch_first=True)  # 64 → 48
        self.drop2 = nn.Dropout(dropout)
        self.lstm3 = nn.LSTM(hidden_dim * 3 // 4, hidden_dim // 2, batch_first=True)  # 48 → 32
        self.fc = nn.Linear(hidden_dim // 2, 1)
    def forward(self, x):
        x, _ = self.lstm1(x); x = self.drop1(x)
        x, _ = self.lstm2(x); x = self.drop2(x)
        x, _ = self.lstm3(x)
        return self.fc(x[:, -1, :])

class CNN1DModel(nn.Module):
    def __init__(self, input_dim, seq_len=WINDOW_SIZE):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(input_dim, 64, kernel_size=7, padding=3),
            nn.ReLU(), nn.BatchNorm1d(64),
            nn.Conv1d(64, 64, kernel_size=5, padding=2),
            nn.ReLU(), nn.BatchNorm1d(64),
            nn.Conv1d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(), nn.AdaptiveAvgPool1d(1)
        )
        self.fc = nn.Linear(64, 1)
    def forward(self, x):
        x = x.permute(0, 2, 1)  # (B, C, T)
        x = self.conv(x).squeeze(-1)
        return self.fc(x)

class TCNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, dilation, dropout=0.1):
        super().__init__()
        padding = (kernel_size - 1) * dilation
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size, padding=padding, dilation=dilation)
        self.bn1 = nn.BatchNorm1d(out_ch)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size, padding=padding, dilation=dilation)
        self.bn2 = nn.BatchNorm1d(out_ch)
        self.drop = nn.Dropout(dropout)
        self.residual = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
        self.relu = nn.ReLU()
    def forward(self, x):
        res = self.residual(x)
        out = self.relu(self.bn1(self.conv1(x)[:, :, :x.size(2)]))
        out = self.drop(out)
        out = self.relu(self.bn2(self.conv2(out)[:, :, :x.size(2)]))
        out = self.drop(out)
        return self.relu(out + res)

class TCNModel(nn.Module):
    def __init__(self, input_dim, channels=[64, 64, 64], kernel_size=5):  # kernel 3 → 5
        super().__init__()
        layers = []
        in_ch = input_dim
        for i, out_ch in enumerate(channels):
            layers.append(TCNBlock(in_ch, out_ch, kernel_size, dilation=2**i))
            in_ch = out_ch
        self.tcn = nn.Sequential(*layers)
        self.fc = nn.Linear(channels[-1], 1)
    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.tcn(x)
        x = x[:, :, -1]
        return self.fc(x)

class TransformerModel(nn.Module):
    def __init__(self, input_dim, d_model=64, nhead=2, num_layers=2, dropout=0.2):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_enc = nn.Parameter(torch.randn(1, WINDOW_SIZE, d_model) * 0.1)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(d_model, 1)
    def forward(self, x):
        x = self.input_proj(x) + self.pos_enc[:, :x.size(1), :]
        x = self.encoder(x)
        return self.fc(x[:, -1, :])

class CNNLSTMModel(nn.Module):
    def __init__(self, input_dim, cnn_out=64, lstm_hidden=64, dropout=0.2):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(input_dim, cnn_out, kernel_size=5, padding=2),
            nn.ReLU(), nn.BatchNorm1d(cnn_out), nn.Dropout(dropout),
            nn.Conv1d(cnn_out, cnn_out, kernel_size=3, padding=1),
            nn.ReLU()
        )
        self.lstm = nn.LSTM(cnn_out, lstm_hidden, batch_first=True,
                            num_layers=2, dropout=dropout)  # 1 → 2 layers + dropout
        self.fc = nn.Linear(lstm_hidden, 1)
    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.cnn(x).permute(0, 2, 1)
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

class AttentionLSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=2,
                            batch_first=True, dropout=dropout)
        self.attn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2), nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1)
        )
        self.fc = nn.Linear(hidden_dim, 1)
    def forward(self, x):
        lstm_out, _ = self.lstm(x)  # (B, T, H)
        attn_weights = torch.softmax(self.attn(lstm_out), dim=1)  # (B, T, 1)
        context = (lstm_out * attn_weights).sum(dim=1)  # (B, H)
        return self.fc(context)

class MLPModel(nn.Module):
    def __init__(self, input_dim, seq_len=WINDOW_SIZE, hidden_dims=[64, 64, 64]):
        super().__init__()
        flat_dim = input_dim * seq_len
        layers = []
        in_dim = flat_dim
        for h in hidden_dims:
            layers += [nn.Linear(in_dim, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(0.2)]
            in_dim = h
        layers.append(nn.Linear(in_dim, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x.view(x.size(0), -1))

print("✓ 10 deep-learning architectures defined")

In [ ]:
# ============================================================
# 6.2  TRAINING — DEEP-LEARNING SCREENING
# ============================================================
dl_results = OrderedDict()

dl_architectures = {
    'LSTM':          lambda: LSTMModel(n_features),
    'GRU':           lambda: GRUModel(n_features),
    'BiLSTM':        lambda: BiLSTMModel(n_features),
    'StackedLSTM':   lambda: StackedLSTMModel(n_features),
    'CNN1D':         lambda: CNN1DModel(n_features),
    'TCN':           lambda: TCNModel(n_features),
    'Transformer':   lambda: TransformerModel(n_features),
    'CNN_LSTM':      lambda: CNNLSTMModel(n_features),
    'AttentionLSTM': lambda: AttentionLSTMModel(n_features),
    'MLP':           lambda: MLPModel(n_features),
}

for name, model_fn in dl_architectures.items():
    print(f"\nTraining: {name}...")
    torch.manual_seed(RANDOM_SEED)
    model = model_fn()
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Parameters: {n_params:,}")

    with timer_context() as t:
        model, history = train_dl_model(model, train_loader, val_loader)

    # Validation predictions
    y_val_pred = predict_dl(model, val_loader, scaler_y)
    # Align lengths (the first WINDOW_SIZE validation hours have no prediction)
    y_val_actual = y_val.values[WINDOW_SIZE:WINDOW_SIZE+len(y_val_pred)]

    dl_results[name] = {
        'model': model,
        'val_metrics': compute_metrics(y_val_actual, y_val_pred[:len(y_val_actual)]),
        'train_time': t.elapsed,
        'n_params': n_params,
        'history': history,
        'type': 'dl'
    }
    print(f"  MAPE: {dl_results[name]['val_metrics']['MAPE']:.4f}% | "
          f"RMSE: {dl_results[name]['val_metrics']['RMSE']:.2f} | "
          f"Epochs: {len(history['train_loss'])} | Time: {t.elapsed:.1f}s")
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

dl_screening_df = print_screening_results(dl_results, 'Deep Learning')

The next cell only releases cached GPU memory before the next stage.

In [ ]:
torch.cuda.empty_cache()

---
## 7. Selection of the models for HPO

The first cell below ranks each family by validation MAPE and lists its **top-3 models**. The second cell then **overrides that automatic selection with a manual choice of one model per family**, which is the selection actually used in the rest of the notebook:

| Family | Selected model |
|---|---|
| Statistical | Lasso |
| Machine learning | AdaBoost |
| Deep learning | Transformer |

The variable names `top3_stat`, `top3_ml` and `top3_dl` are kept for both cells; after the second cell each of them holds a single model. Hence the final comparison (Sections 8–11) involves the MoE and these three tuned baselines.

In [ ]:
# ============================================================
# 7.1  AUTOMATIC TOP-3 SELECTION (by validation MAPE)
# ============================================================
def select_top_k(results_dict, k=3):
    """Select the k models with the lowest validation MAPE."""
    ranked = sorted(results_dict.items(),
                    key=lambda x: x[1]['val_metrics']['MAPE'])
    return OrderedDict(ranked[:k])

top3_stat = select_top_k(stat_results, k=3)
top3_ml   = select_top_k(ml_results,   k=3)
top3_dl   = select_top_k(dl_results,   k=3)

print("=" * 70)
print("TOP-3 MODELS PER FAMILY")
print("=" * 70)
for family, top3 in [('Statistical', top3_stat), ('ML', top3_ml), ('DL', top3_dl)]:
    print(f"\n  {family}:")
    for i, (name, r) in enumerate(top3.items(), 1):
        print(f"    {i}. {name:20s} — MAPE: {r['val_metrics']['MAPE']:.4f}%")

In [ ]:
# ============================================================
# 7.2  MANUAL SELECTION (overrides the automatic top-3)
# ============================================================
def select_models_by_name(results_dict, selected_models):
    """Select specific models by exact name."""
    return OrderedDict(
        (name, results_dict[name])
        for name in selected_models
        if name in results_dict
    )

top3_stat = select_models_by_name(stat_results, ["Lasso"])
top3_ml   = select_models_by_name(ml_results,   ["AdaBoost"])
top3_dl   = select_models_by_name(dl_results,   ["Transformer"])

print("=" * 70)
print("MODELS SELECTED PER FAMILY (manual selection)")
print("=" * 70)
for family, top3 in [('Statistical', top3_stat), ('ML', top3_ml), ('DL', top3_dl)]:
    print(f"\n  {family}:")
    for i, (name, r) in enumerate(top3.items(), 1):
        print(f"    {i}. {name:20s} — MAPE: {r['val_metrics']['MAPE']:.4f}%")

---
## 8. Hyperparameter optimization (HPO) — random search

According to **Bergstra & Bengio (2012)**, random search is more efficient than grid search when only a subset of the hyperparameters really matters, which is the typical case. With **$n=60$** random evaluations, the probability that at least one of them falls in the best 5% of the search space is

$$P(\text{at least one sample in the top 5\%}) = 1 - (1 - 0.05)^{60} \approx 0.954$$

- **Feature-based models (statistical and machine learning)**: `RandomizedSearchCV` with `N_RANDOM_SEARCH_ITER = 60` iterations (10 for Random Forest, for computational cost), scored by MAPE with `TimeSeriesSplit` (3 folds) on the concatenated train + validation data, so that every validation fold comes after its training fold. The best configuration is refitted on train + validation and evaluated on the test split.
- **Deep-learning models**: a manual random search of 20 trials (Section 8.3), because the PyTorch models are not scikit-learn estimators.
- **Univariate time-series models**: no HPO is implemented; if one of them were selected, its screening model would be kept.

### 8.1 Search spaces

Search spaces of the feature-based models. Continuous hyperparameters that span several orders of magnitude (regularization strengths, learning rates, SVR `C` and `epsilon`) are sampled log-uniformly; the others are sampled uniformly or from discrete lists. Spaces are defined for all feature-based models, although only the selected ones are tuned.

In [ ]:
# ============================================================
# 8.1  SEARCH SPACES
# ============================================================
from scipy.stats import uniform, randint, loguniform

# TimeSeriesSplit for time-ordered cross-validation
tscv = TimeSeriesSplit(n_splits=N_CV_FOLDS)

# --- Search spaces of the feature-based models (linear statistical + ML) ---
SEARCH_SPACES = {
    # Feature-based statistical models
    'OLS': None,  # No hyperparameters
    'Ridge': {
        'alpha': loguniform(1e-4, 1e4),
    },
    'Lasso': {
        'alpha': loguniform(1e-4, 1e2),
        'max_iter': [5000, 10000],
    },
    'ElasticNet': {
        'alpha': loguniform(1e-4, 1e2),
        'l1_ratio': uniform(0.01, 0.99),
        'max_iter': [5000, 10000],
    },
    'BayesianRidge': {
        'alpha_1': loguniform(1e-8, 1e-2),
        'alpha_2': loguniform(1e-8, 1e-2),
        'lambda_1': loguniform(1e-8, 1e-2),
        'lambda_2': loguniform(1e-8, 1e-2),
    },
    'HuberRegressor': {
        'epsilon': uniform(1.01, 2.0),
        'alpha': loguniform(1e-5, 1e-1),
        'max_iter': [500, 1000, 2000],
    },
    # ML
    'RandomForest': {
        'n_estimators': randint(100, 500),
        'max_depth': randint(5, 30),
        'min_samples_split': randint(2, 20),
        'min_samples_leaf': randint(1, 10),
        'max_features': ['sqrt', 'log2', 0.5, 0.8, None],
    },
    'GradientBoosting': {
        'n_estimators': randint(100, 500),
        'max_depth': randint(3, 12),
        'learning_rate': loguniform(0.005, 0.3),
        'subsample': uniform(0.6, 0.4),
        'min_samples_split': randint(2, 20),
    },
    'XGBoost': {
        'n_estimators': randint(100, 500),
        'max_depth': randint(3, 12),
        'learning_rate': loguniform(0.005, 0.3),
        'subsample': uniform(0.6, 0.4),
        'colsample_bytree': uniform(0.5, 0.5),
        'reg_alpha': loguniform(1e-3, 10),
        'reg_lambda': loguniform(1e-3, 10),
    },
    'LightGBM': {
        'n_estimators': randint(100, 500),
        'max_depth': randint(3, 15),
        'learning_rate': loguniform(0.005, 0.3),
        'num_leaves': randint(20, 100),
        'subsample': uniform(0.6, 0.4),
        'colsample_bytree': uniform(0.5, 0.5),
        'reg_alpha': loguniform(1e-3, 10),
        'reg_lambda': loguniform(1e-3, 10),
    },
    'ExtraTrees': {
        'n_estimators': randint(100, 500),
        'max_depth': randint(5, 30),
        'min_samples_split': randint(2, 20),
        'min_samples_leaf': randint(1, 10),
    },
    'AdaBoost': {
        'n_estimators': randint(50, 300),
        'learning_rate': loguniform(0.01, 1.0),
    },
    'SVR': {
        'svr__C': loguniform(1, 1000),
        'svr__epsilon': loguniform(0.01, 1),
        'svr__gamma': ['scale', 'auto'],
    },
    'KNN': {
        'n_neighbors': randint(3, 30),
        'weights': ['uniform', 'distance'],
        'p': [1, 2],
    },
    'Bagging': {
        'n_estimators': randint(20, 150),
        'max_samples': uniform(0.5, 0.5),
        'max_features': uniform(0.5, 0.5),
    },
    'DecisionTree': {
        'max_depth': randint(5, 30),
        'min_samples_split': randint(2, 20),
        'min_samples_leaf': randint(1, 10),
    },
}

print("✓ Search spaces defined for", len(SEARCH_SPACES), "models")

### 8.2 HPO of the feature-based models

Each selected statistical or machine-learning model is rebuilt with its seed and parallelism settings and tuned with `RandomizedSearchCV` on train + validation. The refitted best estimator is then evaluated once on the test split.

In [ ]:
# ============================================================
# 8.2  HPO — FEATURE-BASED MODELS (statistical + ML)
# ============================================================
hpo_results = OrderedDict()

# Collect the selected statistical and ML models that are feature-based
top3_feature_based = {}
for name, r in list(top3_stat.items()) + list(top3_ml.items()):
    if r['type'] == 'feature_based' and name in SEARCH_SPACES:
        top3_feature_based[name] = r

print(f"Feature-based models for HPO: {list(top3_feature_based.keys())}")
print(f"Using RandomizedSearchCV with n_iter={N_RANDOM_SEARCH_ITER}, CV={N_CV_FOLDS} folds\n")

for name, r in top3_feature_based.items():
    space = SEARCH_SPACES.get(name)
    if space is None:
        print(f"  {name}: no hyperparameters → screening model kept")
        hpo_results[name] = r
        continue

    print(f"  Optimizing: {name} ({N_RANDOM_SEARCH_ITER} iterations)...")
    base_model = r['model']

    # For pipelines (SVR), rebuild the full pipeline
    if isinstance(base_model, Pipeline):
        estimator = Pipeline([
            ('scaler', StandardScaler()),
            ('svr', SVR(kernel='rbf'))
        ])
    else:
        estimator = base_model.__class__(**{
            k: v for k, v in base_model.get_params().items()
            if k in ['random_state', 'n_jobs', 'verbose', 'verbosity']
            and k in base_model.get_params()
        })
        # Re-set the seed and parallelism parameters
        try: estimator.set_params(random_state=RANDOM_SEED)
        except: pass
        try: estimator.set_params(n_jobs=N_JOBS)
        except: pass

    with timer_context() as t:
        search = RandomizedSearchCV(
            estimator, space,
            n_iter=N_RANDOM_SEARCH_ITER if name != 'RandomForest' else 10,
            scoring='neg_mean_absolute_percentage_error',
            cv=tscv, random_state=RANDOM_SEED, n_jobs=N_JOBS,
            verbose=0, refit=True
        )
        search.fit(X_trainval, y_trainval)

    # Evaluate on the test split
    best_model = search.best_estimator_
    preds_test = best_model.predict(X_test)
    test_metrics = compute_metrics(y_test, preds_test)

    hpo_results[name] = {
        'model': best_model,
        'best_params': search.best_params_,
        'val_metrics': r['val_metrics'],  # screening metrics kept for reference
        'test_metrics': test_metrics,
        'train_time': t.elapsed,
        'best_cv_score': -search.best_score_,
        'type': r['type']
    }
    print(f"    ✓ Best CV MAPE: {-search.best_score_:.4f}% | Test MAPE: {test_metrics['MAPE']:.4f}%")
    print(f"    Best params: {search.best_params_}")
    print(f"    HPO time: {t.elapsed:.1f}s\n")

# Univariate time-series models (standard RandomizedSearchCV does not apply)
for name, r in top3_stat.items():
    if r['type'] == 'ts_univariate' and name not in hpo_results:
        print(f"  {name} (univariate TS): HPO not implemented → screening model kept")
        hpo_results[name] = r

print(f"\n✓ HPO completed for {len(hpo_results)} feature-based/statistical models")

### 8.3 HPO of the deep-learning models

Manual random search over hidden size, number of layers, dropout, learning rate and batch size (20 trials per architecture). Each trial is trained on the training split for at most 30 epochs (early-stopping patience 5) and scored by its MAPE on the validation split. Some architectures do not use every sampled hyperparameter (for example, CNN-1D ignores them all and the stacked LSTM has a fixed number of layers); see `create_dl_model_with_params`.

The best configuration is then retrained from scratch on train + validation (at most 50 epochs, patience 7) and evaluated on the test split.

In [ ]:
# ============================================================
# 8.3  HPO — DEEP-LEARNING MODELS (manual random search)
# ============================================================
# Manual random search over the key deep-learning hyperparameters

DL_SEARCH_SPACES = {
    'hidden_dim': [32, 64, 128, 256],
    'num_layers': [1, 2, 3],
    'dropout': [0.1, 0.2, 0.3, 0.4],
    'lr': [5e-4, 1e-3, 2e-3, 5e-3],
    'batch_size': [128, 256, 512],
}

dl_hpo_results = OrderedDict()

# Map each architecture name to a constructor with the sampled hyperparameters
def create_dl_model_with_params(arch_name, input_dim, hidden_dim, num_layers, dropout):
    """Build a deep-learning model with the given hyperparameters."""
    if arch_name == 'LSTM':
        return LSTMModel(input_dim, hidden_dim, num_layers, dropout)
    elif arch_name == 'GRU':
        return GRUModel(input_dim, hidden_dim, num_layers, dropout)
    elif arch_name == 'BiLSTM':
        return BiLSTMModel(input_dim, hidden_dim // 2, num_layers, dropout)
    elif arch_name == 'StackedLSTM':
        return StackedLSTMModel(input_dim, hidden_dim, dropout)
    elif arch_name == 'CNN1D':
        return CNN1DModel(input_dim)
    elif arch_name == 'TCN':
        return TCNModel(input_dim, channels=[hidden_dim, hidden_dim*2, hidden_dim])
    elif arch_name == 'Transformer':
        # nhead must divide d_model
        d_model = max(32, hidden_dim)
        nhead = 4 if d_model % 4 == 0 else 2
        return TransformerModel(input_dim, d_model=d_model, nhead=nhead,
                                num_layers=num_layers, dropout=dropout)
    elif arch_name == 'CNN_LSTM':
        return CNNLSTMModel(input_dim, cnn_out=hidden_dim, lstm_hidden=hidden_dim)
    elif arch_name == 'AttentionLSTM':
        return AttentionLSTMModel(input_dim, hidden_dim, dropout)
    elif arch_name == 'MLP':
        return MLPModel(input_dim, WINDOW_SIZE, [hidden_dim*2, hidden_dim, hidden_dim//2])
    else:
        raise ValueError(f"Unknown architecture: {arch_name}")

N_DL_RANDOM_TRIALS = 20  # Reduced for computational cost; increase if a GPU is available

for arch_name in top3_dl.keys():
    print(f"\nHPO for {arch_name} ({N_DL_RANDOM_TRIALS} trials)...")
    best_mape = float('inf')
    best_config = None
    best_model_state = None
    trials_log = []

    for trial in range(N_DL_RANDOM_TRIALS):
        # Sample hyperparameters
        config = {
            'hidden_dim': np.random.choice(DL_SEARCH_SPACES['hidden_dim']),
            'num_layers': np.random.choice(DL_SEARCH_SPACES['num_layers']),
            'dropout': np.random.choice(DL_SEARCH_SPACES['dropout']),
            'lr': np.random.choice(DL_SEARCH_SPACES['lr']),
            'batch_size': np.random.choice(DL_SEARCH_SPACES['batch_size']),
        }

        torch.manual_seed(RANDOM_SEED + trial)
        model = create_dl_model_with_params(
            arch_name, n_features,
            int(config['hidden_dim']),
            int(config['num_layers']),
            config['dropout']
        )

        # Rebuild the loaders with the sampled batch size
        tl = DataLoader(train_ds, batch_size=int(config['batch_size']),
                        shuffle=False, drop_last=True)
        vl = DataLoader(val_ds, batch_size=int(config['batch_size']), shuffle=False)

        try:
            model, hist = train_dl_model(model, tl, vl, epochs=30,
                                         lr=config['lr'], patience=5)
            y_pred = predict_dl(model, vl, scaler_y)
            y_actual = y_val.values[WINDOW_SIZE:WINDOW_SIZE+len(y_pred)]
            mets = compute_metrics(y_actual, y_pred[:len(y_actual)])

            trials_log.append({**config, 'MAPE': mets['MAPE'], 'trial': trial})

            if mets['MAPE'] < best_mape:
                best_mape = mets['MAPE']
                best_config = config
                best_model_state = copy.deepcopy(model.state_dict())
                best_model_arch = model

            if (trial + 1) % 5 == 0:
                print(f"    Trial {trial+1}/{N_DL_RANDOM_TRIALS} | "
                      f"Current MAPE: {mets['MAPE']:.4f}% | Best: {best_mape:.4f}%")
        except Exception as e:
            print(f"    Trial {trial+1} failed: {e}")
            continue

        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    # Retrain the best configuration with more epochs
    print(f"  ✓ Best config: {best_config} → MAPE: {best_mape:.4f}%")

    if best_model_state is not None:
        # Retrain on train + validation (all data before the test split)
        X_tv_sc = scaler_X.transform(X_trainval)
        y_tv_sc = scaler_y.transform(y_trainval.values.reshape(-1,1)).flatten()
        tv_ds = TimeSeriesDataset(X_tv_sc, y_tv_sc, WINDOW_SIZE)
        X_test_sc = scaler_X.transform(X_test)
        y_test_sc = scaler_y.transform(y_test.values.reshape(-1,1)).flatten()
        test_ds = TimeSeriesDataset(X_test_sc, y_test_sc, WINDOW_SIZE)

        tv_loader = DataLoader(tv_ds, batch_size=int(best_config['batch_size']),
                               shuffle=False, drop_last=True)
        test_loader_dl = DataLoader(test_ds, batch_size=int(best_config['batch_size']),
                                    shuffle=False)

        torch.manual_seed(RANDOM_SEED)
        final_model = create_dl_model_with_params(
            arch_name, n_features,
            int(best_config['hidden_dim']),
            int(best_config['num_layers']),
            best_config['dropout']
        )
        with timer_context() as t_retrain:
            final_model, _ = train_dl_model(
                final_model, tv_loader, test_loader_dl,
                epochs=DL_EPOCHS, lr=best_config['lr'], patience=DL_PATIENCE
            )

        # Test predictions
        test_preds = predict_dl(final_model, test_loader_dl, scaler_y)
        y_test_actual = y_test.values[WINDOW_SIZE:WINDOW_SIZE+len(test_preds)]
        test_mets = compute_metrics(y_test_actual, test_preds[:len(y_test_actual)])

        dl_hpo_results[arch_name] = {
            'model': final_model,
            'best_params': best_config,
            'test_metrics': test_mets,
            'val_metrics': top3_dl[arch_name]['val_metrics'],
            'train_time': t_retrain.elapsed,
            'trials_log': trials_log,
            'type': 'dl'
        }
        print(f"  ✓ Test MAPE: {test_mets['MAPE']:.4f}% | "
              f"RMSE: {test_mets['RMSE']:.2f} | Time: {t_retrain.elapsed:.1f}s")

print(f"\n✓ Deep-learning HPO completed for {len(dl_hpo_results)} models")

In [ ]:
torch.cuda.empty_cache()

### 8.4 Export of intermediate results

The screening tables (CSV) and the screening, selection and HPO results (pickle, including the fitted models) are saved to `BENCH_DIR` so that the later analyses can be inspected or reproduced without re-training.

In [ ]:
import pickle
import pandas as pd
from collections import OrderedDict

# List of variables to export and their types
variables_to_export = {
    'stat_screening_df': {'obj': stat_screening_df, 'type': 'dataframe'},
    'ml_screening_df': {'obj': ml_screening_df, 'type': 'dataframe'},
    'dl_screening_df': {'obj': dl_screening_df, 'type': 'dataframe'},
    'stat_results': {'obj': stat_results, 'type': 'pickle'},
    'ml_results': {'obj': ml_results, 'type': 'pickle'},
    'dl_results': {'obj': dl_results, 'type': 'pickle'},
    'top3_stat': {'obj': top3_stat, 'type': 'pickle'},
    'top3_ml': {'obj': top3_ml, 'type': 'pickle'},
    'top3_dl': {'obj': top3_dl, 'type': 'pickle'},
    'hpo_results': {'obj': hpo_results, 'type': 'pickle'},
    'dl_hpo_results': {'obj': dl_hpo_results, 'type': 'pickle'},
}

print(f"Exporting variables to: {BENCH_DIR}")

for var_name, var_info in variables_to_export.items():
    obj = var_info['obj']
    obj_type = var_info['type']

    try:
        if obj_type == 'dataframe':
            filepath = BENCH_DIR / f'{var_name}.csv'
            obj.to_csv(filepath, index=False)
            print(f"  ✓ Exported {var_name} as CSV to {filepath}")
        elif obj_type == 'pickle':
            filepath = BENCH_DIR / f'{var_name}.pkl'
            with open(filepath, 'wb') as f:
                pickle.dump(obj, f)
            print(f"  ✓ Exported {var_name} as PKL to {filepath}")
        else:
            print(f"  ✗ Unsupported object type for {var_name}: {type(obj)}")
    except Exception as e:
        print(f"  ❌ Error while exporting {var_name}: {e}")

print("\n✓ Variable export completed.")

---
## 9. Final evaluation on the test set

The MoE and every tuned baseline are evaluated on the same test split with the same metrics (MAE, RMSE, MAPE, R²). The results are collected in the master table `summary_df` (sorted by test MAPE) and saved to `benchmark_summary.csv`.

- **Feature-based models**: test metrics computed in Section 8.2 on all test observations; inference time is measured here.
- **Deep-learning models**: predictions and inference time measured here on the test split; the first `WINDOW_SIZE` = 168 test hours have no prediction (Section 6), so their metrics cover the remaining test hours.
- **Univariate time-series models** (if selected): their validation metrics are used as a proxy for the test metrics.
- **MoE**: test metrics from Section 3. Its training and inference times are entered as fixed values in the code (`train_time = 20.7954` s, `infer_time = 0.2859` s) instead of the values measured in this session.

Model names carry a suffix: `(Prop)` for the proposed MoE, `(Opt)` for tuned baselines and `(TS)` for univariate time-series models.

In [ ]:
# ============================================================
# 9. FINAL EVALUATION — ALL MODELS ON THE TEST SET
# ============================================================
final_results = OrderedDict()

# --- MoE ---
final_results['MoE (Prop)'] = {
    'family': 'MoE',
    'test_metrics': moe_metrics_test,
    'train_time': 20.7954,  # fixed value (measured value: moe_train_time)
    'infer_time': 0.2859,   # fixed value (measured value: moe_infer_time)
}

# --- Tuned feature-based models ---
for name, r in hpo_results.items():
    if r['type'] == 'feature_based':
        model = r['model']
        with timer_context() as t_infer:
            preds = model.predict(X_test)
        test_mets = compute_metrics(y_test, preds) if 'test_metrics' not in r else r['test_metrics']
        final_results[f'{name} (Opt)'] = {
            'family': 'Statistical' if name in stat_results else 'Machine Learning',
            'test_metrics': test_mets,
            'train_time': r['train_time'],
            'infer_time': t_infer.elapsed,
            'predictions': preds,
        }
    elif r['type'] == 'ts_univariate':
        # For univariate time-series models, validation metrics are used as a proxy
        final_results[f'{name} (TS)'] = {
            'family': 'Statistical',
            'test_metrics': r.get('test_metrics', r['val_metrics']),
            'train_time': r['train_time'],
            'infer_time': 0,  # inference of the classical TS models is treated as negligible
        }

# --- Tuned deep-learning models ---
# Test loader for the deep-learning models, used to measure inference time
X_test_sc_dl = scaler_X.transform(X_test)
y_test_sc_dl = scaler_y.transform(y_test.values.reshape(-1,1)).flatten()  # required by the Dataset; not used by predict_dl
test_ds_dl = TimeSeriesDataset(X_test_sc_dl, y_test_sc_dl, WINDOW_SIZE)
test_loader_dl = DataLoader(test_ds_dl, batch_size=DL_BATCH, shuffle=False)

for name, r in dl_hpo_results.items():
    model = r['model']
    with timer_context() as t_infer_dl:
        # predict_dl produces the predictions; the timer measures inference time
        preds_dl = predict_dl(model, test_loader_dl, scaler_y)
    # Align the actual values with the predictions (first WINDOW_SIZE hours have none)
    y_test_actual_dl = y_test.values[WINDOW_SIZE : WINDOW_SIZE + len(preds_dl)]

    final_results[f'{name} (Opt)'] = {
        'family': 'Deep Learning',
        'test_metrics': compute_metrics(y_test_actual_dl, preds_dl[:len(y_test_actual_dl)]),
        'train_time': r['train_time'],
        'infer_time': t_infer_dl.elapsed,
    }

# --- Summary table (column names are kept as written to benchmark_summary.csv) ---
summary_rows = []
for name, r in final_results.items():
    m = r['test_metrics']
    summary_rows.append({
        'Model': name,
        'Family': r['family'],
        'MAPE (%)': m['MAPE'],
        'RMSE': m['RMSE'],
        'MAE': m['MAE'],
        'R²': m['R2'],
        'Train Time (s)': r['train_time'],
        'Infer Time (s)': r.get('infer_time', np.nan),
    })

summary_df = pd.DataFrame(summary_rows).sort_values('MAPE (%)')
print("=" * 100)
print("  FINAL EVALUATION ON THE TEST SET — ALL MODELS")
print("=" * 100)
print(summary_df.to_string(index=False, float_format='{:.4f}'.format))

# Save
summary_df.to_csv(BENCH_DIR / 'benchmark_summary.csv', index=False)
print(f"\n✓ Results saved to {BENCH_DIR / 'benchmark_summary.csv'}")

---
## 10. Comparative analysis

The test-set results of Section 9 are analysed along four dimensions: predictive performance (10.1), computational efficiency (10.2), interpretability (10.3) and statistical significance of the differences (10.4). Section 10.5 adds a robustness stress test on the excluded COVID-19 period.

### 10.1 Predictive performance

Horizontal bar charts of MAPE, RMSE, MAE and R² on the test set for every model, coloured by family, followed by a radar chart of the six best models (by test MAPE). In the radar chart each metric is min-max normalized across the models so that 1 is always the best value (MAPE, RMSE and MAE are inverted).

In [ ]:
# ============================================================
# 10.1  PREDICTIVE PERFORMANCE
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Predictive Performance Comparison — Test Set', fontsize=16, fontweight='bold')

# Colour palette by family (keys are the family labels stored in the results)
family_colors = {'MoE': '#E63946', 'Statistical': '#457B9D',
                 'Machine Learning': '#2A9D8F', 'Deep Learning': '#E9C46A'}
colors = [family_colors.get(r['family'], '#999') for _, r in final_results.items()]
names = list(final_results.keys())

metrics_to_plot = ['MAPE (%)', 'RMSE', 'MAE', 'R²']
for ax, metric in zip(axes.flatten(), metrics_to_plot):
    values = summary_df[metric].values
    bars = ax.barh(summary_df['Model'], values, color=[
        family_colors.get(f, '#999') for f in summary_df['Family']
    ], edgecolor='white', linewidth=0.5)
    ax.set_xlabel(metric)
    ax.set_title(metric, fontweight='bold')
    ax.invert_yaxis()
    # Annotate values
    for bar, val in zip(bars, values):
        ax.text(bar.get_width(), bar.get_y() + bar.get_height()/2,
                f'  {val:.4f}', va='center', fontsize=8)

# Legend
patches = [mpatches.Patch(color=c, label=f) for f, c in family_colors.items()]
fig.legend(handles=patches, loc='lower center', ncol=4, fontsize=11,
           bbox_to_anchor=(0.5, -0.02))
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.savefig(BENCH_DIR / 'fig_predictive_performance.png', bbox_inches='tight')
plt.show()
print("✓ Figure saved: fig_predictive_performance.png")

In [ ]:
# ============================================================
# 10.1b  RADAR CHART — multi-metric profile of each model
# ============================================================
from matplotlib.patches import FancyBboxPatch

# Normalize the metrics for the radar chart (0-1, where 1 is best)
radar_df = summary_df.copy()
for col in ['MAPE (%)', 'RMSE', 'MAE']:
    vmin, vmax = radar_df[col].min(), radar_df[col].max()
    if vmax > vmin:
        radar_df[col + '_norm'] = 1 - (radar_df[col] - vmin) / (vmax - vmin)  # inverted: lower is better
    else:
        radar_df[col + '_norm'] = 1.0
radar_df['R²_norm'] = (radar_df['R²'] - radar_df['R²'].min()) / (
    radar_df['R²'].max() - radar_df['R²'].min()) if radar_df['R²'].max() > radar_df['R²'].min() else 1.0

categories = ['MAPE', 'RMSE', 'MAE', 'R²']
norm_cols = ['MAPE (%)_norm', 'RMSE_norm', 'MAE_norm', 'R²_norm']

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
angles = np.linspace(0, 2*np.pi, len(categories), endpoint=False).tolist()
angles += angles[:1]

for i, row in radar_df.head(6).iterrows():  # Top 6 models
    values = [row[c] for c in norm_cols]
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=row['Model'])
    ax.fill(angles, values, alpha=0.1)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=12)
ax.set_title('Multi-Metric Profile — Top 6 Models', fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=9)
plt.tight_layout()
plt.savefig(BENCH_DIR / 'fig_radar_comparison.png', bbox_inches='tight')
plt.show()

### 10.2 Computational efficiency

Training time (for the tuned baselines, the duration of the HPO search or of the final deep-learning retraining, as recorded in Section 8) and inference time on the test split, each shown as a bar chart and as an accuracy-vs-time trade-off scatter plot. The training throughput table divides the number of train + validation samples by the training time.

In [ ]:
# ============================================================
# 10.2a  COMPUTATIONAL EFFICIENCY — TRAINING TIME
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Training time
ax = axes[0]
train_times = summary_df.sort_values('Train Time (s)')
colors_t = [family_colors.get(f, '#999') for f in train_times['Family']]
ax.barh(train_times['Model'], train_times['Train Time (s)'],
        color=colors_t, edgecolor='white')
ax.set_xlabel('Training Time (s)')
ax.set_title('Training Time', fontweight='bold')
ax.invert_yaxis()
for i, (_, row) in enumerate(train_times.iterrows()):
    ax.text(row['Train Time (s)'], i, f"  {row['Train Time (s)']:.1f}s",
            va='center', fontsize=8)

# Trade-off: MAPE vs. training time
ax = axes[1]
for family, color in family_colors.items():
    mask = summary_df['Family'] == family
    sub = summary_df[mask]
    ax.scatter(sub['Train Time (s)'], sub['MAPE (%)'],
               c=color, s=120, label=family, edgecolors='black', zorder=5)
    for _, row in sub.iterrows():
        ax.annotate(row['Model'].split(' (')[0], (row['Train Time (s)'], row['MAPE (%)']),
                    fontsize=7, ha='left', va='bottom', xytext=(5, 5),
                    textcoords='offset points')
ax.set_xlabel('Training Time (s)')
ax.set_ylabel('MAPE (%)')
ax.set_title('Trade-off: Accuracy vs. Efficiency', fontweight='bold')
ax.legend()

plt.tight_layout()
plt.savefig(BENCH_DIR / 'fig_computational_efficiency_train.png', bbox_inches='tight')
plt.show()
print("✓ Figure saved: fig_computational_efficiency_train.png")

In [ ]:
# ============================================================
# 10.2b  COMPUTATIONAL EFFICIENCY — INFERENCE TIME
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Inference time
ax = axes[0]
train_times = summary_df.sort_values('Infer Time (s)')
colors_t = [family_colors.get(f, '#999') for f in train_times['Family']]
ax.barh(train_times['Model'], train_times['Infer Time (s)'],
        color=colors_t, edgecolor='white')
ax.set_xlabel('Inference Time (s)')
ax.set_title('Inference Time', fontweight='bold')
ax.invert_yaxis()
for i, (_, row) in enumerate(train_times.iterrows()):
    ax.text(row['Infer Time (s)'], i, f"  {row['Infer Time (s)']:.1f}s",
            va='center', fontsize=8)

# Trade-off: MAPE vs. inference time
ax = axes[1]
for family, color in family_colors.items():
    mask = summary_df['Family'] == family
    sub = summary_df[mask]
    ax.scatter(sub['Infer Time (s)'], sub['MAPE (%)'],
               c=color, s=120, label=family, edgecolors='black', zorder=5)
    for _, row in sub.iterrows():
        ax.annotate(row['Model'].split(' (')[0], (row['Infer Time (s)'], row['MAPE (%)']),
                    fontsize=7, ha='left', va='bottom', xytext=(5, 5),
                    textcoords='offset points')
ax.set_xlabel('Inference Time (s)')
ax.set_ylabel('MAPE (%)')
ax.set_title('Trade-off: Accuracy vs. Efficiency', fontweight='bold')
ax.legend()

plt.tight_layout()
plt.savefig(BENCH_DIR / 'fig_computational_efficiency_inf.png', bbox_inches='tight')
plt.show()
print("✓ Figure saved: fig_computational_efficiency_inf.png")

In [ ]:
# ============================================================
# 10.2c  SCALABILITY TABLE (training throughput)
# ============================================================
scalability_rows = []
for name, r in final_results.items():
    n_train_samples = len(X_trainval)
    train_time = r['train_time'] if r['train_time'] else 0
    throughput = n_train_samples / train_time if train_time > 0 else np.inf
    scalability_rows.append({
        'Model': name,
        'Family': r['family'],
        'Train Samples': n_train_samples,
        'Train Time (s)': train_time,
        'Throughput (samples/s)': throughput,
        'MAPE (%)': r['test_metrics']['MAPE'],
    })

scalability_df = pd.DataFrame(scalability_rows).sort_values('Throughput (samples/s)', ascending=False)
print("\n" + "="*80)
print("  SCALABILITY — Training throughput")
print("="*80)
print(scalability_df.to_string(index=False, float_format='{:.2f}'.format))
scalability_df.to_csv(BENCH_DIR / 'scalability_analysis.csv', index=False)

### 10.3 Interpretability

Interpretability is analysed in **five complementary ways**. The first one is a qualitative 1–5 scale taken from a state-of-the-art review on electricity demand forecasting (Torres-Sánchez et al.):

| Score | Meaning |
|---------|-------------|
| **1** | Black box with no useful operational explanation |
| **2** | Limited explainability that depends on auxiliary post-hoc techniques (SHAP, LIME) without sufficient traceability |
| **3** | Partial interpretability: internal measures (feature importance, attention weights) but no complete traceability |
| **4** | High interpretability: model components that can be analysed directly (e.g., experts specialized by domain) |
| **5** | Full interpretability: coefficients, rules or components that can be read and audited directly |

#### Analyses
1. **Quantitative 1–5 scale** (10.3.1) — score per model, assigned from its model class.
2. **SHAP values** (10.3.2) — contribution of each feature to the forecast of each tuned baseline, and agreement between the models' SHAP feature rankings.
3. **t-SNE/UMAP of the expert assignment** (10.3.3) — visualization of the MoE routing in feature space and of expert activation by operating context.
4. **Routing stability** (10.3.4) — consistency of the expert assignment for similar samples.
5. **Consolidated summary** (10.3.5).

#### 10.3.1 Quantitative interpretability scale (1–5)

Each model receives a fixed score according to its model class (linear models and classical time-series models 4–5, single decision tree 4, tree ensembles 3, other machine-learning models and most deep-learning models 1–2, MoE 4). The score is looked up by matching the model name, and the figure plots it against the test MAPE to show the interpretability–accuracy trade-off.

In [ ]:
# ============================================================
# 10.3.1  QUANTITATIVE INTERPRETABILITY SCALE (1-5)
# ============================================================
# Scale from the state-of-the-art review article on
# electricity demand forecasting
# ============================================================

def assign_interpretability_score(model_name, family):
    """
    Assign a 1-5 score according to the state-of-the-art scale.

    Criteria:
    - 5: Directly readable coefficients/rules (linear regression, Lasso, Ridge)
    - 4: Components that can be analysed by domain (MoE with experts + explicit gating)
    - 3: Partial internal measures (tree importance, attention weights)
    - 2: Requires post-hoc techniques (SHAP/LIME) to be explained
    - 1: Black box without a useful explanation
    """
    scores = {
        # Linear statistical models → directly auditable coefficients
        'OLS': 5, 'Ridge': 5, 'Lasso': 5, 'ElasticNet': 5,
        'BayesianRidge': 5, 'HuberRegressor': 5,
        # Classical TS models → readable components (trend, seasonality)
        'AutoReg': 5, 'HoltWinters': 5, 'SARIMAX': 4, 'Prophet': 4,
        # Tree-based ML → native feature importance, but no direct coefficients
        'RandomForest': 3, 'GradientBoosting': 3, 'XGBoost': 3,
        'LightGBM': 3, 'ExtraTrees': 3, 'AdaBoost': 3,
        'DecisionTree': 4,  # Single tree → directly readable rules
        # Other ML → require SHAP/LIME
        'SVR': 2, 'KNN': 2, 'Bagging': 2,
        # DL → black box, limited interpretability
        'LSTM': 2, 'GRU': 2, 'BiLSTM': 2, 'StackedLSTM': 1,
        'CNN1D': 1, 'TCN': 1, 'Transformer': 2,  # partial attention weights
        'CNN_LSTM': 1, 'AttentionLSTM': 2, 'MLP': 1,
        # MoE → domain-specialized experts + explicit gating
        'MoE': 4,
    }
    # Match by base name (ignoring suffixes such as "(Opt)", "(DL-Opt)")
    for key in scores:
        if key in model_name:
            return scores[key]
    return 2  # conservative default

# Build the table
interp_rows = []
for name, r in final_results.items():
    family = r['family']
    score = assign_interpretability_score(name, family)

    # Justification for each score
    justifications = {
        5: 'Directly readable and auditable coefficients/components',
        4: 'Model components can be analysed directly by domain',
        3: 'Native feature importance available, but no complete traceability',
        2: 'Requires post-hoc techniques (SHAP, LIME) for explainability',
        1: 'Black box: opaque weights, no direct operational explanation',
    }

    interp_rows.append({
        'Model': name,
        'Family': family,
        'Score (1-5)': score,
        'Justification': justifications[score],
        'MAPE (%)': r['test_metrics']['MAPE'],
    })

interp_score_df = pd.DataFrame(interp_rows).sort_values('Score (1-5)', ascending=False)

print("=" * 110)
print("  QUANTITATIVE INTERPRETABILITY SCALE (1-5)")
print("  Scale: Torres-Sánchez et al. — state-of-the-art review of electricity demand forecasting")
print("=" * 110)
print(interp_score_df.to_string(index=False))

# Figure: interpretability vs. MAPE (trade-off)
fig, ax = plt.subplots(figsize=(12, 7))
for family, color in family_colors.items():
    mask = interp_score_df['Family'] == family
    sub = interp_score_df[mask]
    ax.scatter(sub['Score (1-5)'], sub['MAPE (%)'],
               c=color, s=150, label=family, edgecolors='black', zorder=5, alpha=0.85)
    for _, row in sub.iterrows():
        ax.annotate(row['Model'].split(' (')[0],
                    (row['Score (1-5)'], row['MAPE (%)']),
                    fontsize=7, ha='left', va='bottom',
                    xytext=(6, 4), textcoords='offset points')

ax.set_xlabel('Interpretability (1 = Black Box → 5 = Fully Auditable)', fontsize=12)
ax.set_ylabel('MAPE (%)', fontsize=12)
ax.set_title('Trade-off: Interpretability vs. Predictive Performance', fontsize=14, fontweight='bold')
ax.set_xticks([1, 2, 3, 4, 5])
ax.set_xticklabels(['1\nBlack box', '2\nPost-hoc\n(SHAP/LIME)',
                     '3\nPartial feature\nimportance', '4\nAnalyzable\ncomponents',
                     '5\nFully\nauditable'], fontsize=9)
ax.legend(loc='upper right')
ax.invert_yaxis()  # Lower MAPE = better → top
plt.tight_layout()
plt.savefig(BENCH_DIR / 'fig_interpretability_vs_performance.png', bbox_inches='tight')
plt.show()

interp_score_df.to_csv(BENCH_DIR / 'interpretability_scores.csv', index=False)
print("\n✓ Table and figure saved")

#### 10.3.2 SHAP values — feature contributions per model

SHAP (SHapley Additive exPlanations) quantifies the marginal contribution of each feature to a model's forecast. Because it is model-agnostic, it allows comparing **which information each baseline relies on**. SHAP values are computed on a random sample of 200 test observations, with a background sample of 500 test observations (smaller subsets are used with the slower explainers):

- **Tree-based models** (or any model exposing `feature_importances_`): `TreeExplainer`, with `KernelExplainer` as a fallback if it fails.
- **Pipelines** (e.g., SVR with scaler): `KernelExplainer` (100 background samples, 50 explained samples).
- **Linear models**: `LinearExplainer` with an independent masker, with `KernelExplainer` as a fallback.
- **Deep-learning models**: `KernelExplainer` on a wrapper that receives one feature vector and repeats it over the 168-step input window. This is an approximation: it attributes the forecast to the current feature values only, not to the temporal sequence (50 background samples, 30 explained samples, 100 evaluations per sample).

SHAP values are not computed for the MoE; its interpretability is analysed through its routing (Sections 10.3.3–10.3.4).

Then the mean |SHAP| bar charts of one representative per family and the beeswarm plot of the model in row 2 of the test ranking (`summary_df.iloc[1]`) are plotted, and the agreement between the models' SHAP feature rankings is measured with Spearman's rank correlation.

In [ ]:
import shap
shap.initjs()

In [ ]:
# ============================================================
# 10.3.2a  SHAP VALUES — FEATURE-BASED MODELS
# ============================================================

# Subsampling for computational efficiency
N_SHAP_SAMPLES = 500  # Background
N_SHAP_EXPLAIN = 200  # Samples to explain
np.random.seed(RANDOM_SEED)

shap_background = X_test.sample(n=min(N_SHAP_SAMPLES, len(X_test)), random_state=RANDOM_SEED)
shap_explain    = X_test.sample(n=min(N_SHAP_EXPLAIN, len(X_test)), random_state=RANDOM_SEED + 1)

shap_values_dict = {}

# --- SHAP for the feature-based models (TreeExplainer, LinearExplainer or KernelExplainer) ---
for name, r in hpo_results.items():
    if r['type'] != 'feature_based':
        continue
    model = r['model']
    print(f"SHAP for {name}...", end=" ")

    try:
        # TreeExplainer for tree-based models (much faster)
        if hasattr(model, 'feature_importances_') or name in ['RandomForest', 'GradientBoosting',
            'XGBoost', 'LightGBM', 'ExtraTrees', 'DecisionTree']:
            try:
                explainer = shap.TreeExplainer(model)
                sv = explainer.shap_values(shap_explain)
            except:
                explainer = shap.KernelExplainer(model.predict, shap_background.values[:100])
                sv = explainer.shap_values(shap_explain.values[:50])
        elif isinstance(model, Pipeline):
            # Pipelines with a scaler (SVR, etc.) → KernelExplainer
            explainer = shap.KernelExplainer(model.predict, shap_background.values[:100])
            sv = explainer.shap_values(shap_explain.values[:50])
        else:
            # Linear models → LinearExplainer
            try:
                masker = shap.maskers.Independent(shap_background, max_samples=100)
                explainer = shap.LinearExplainer(model, masker)
                sv = explainer.shap_values(shap_explain)
            except:
                explainer = shap.KernelExplainer(model.predict, shap_background.values[:100])
                sv = explainer.shap_values(shap_explain.values[:50])

        shap_values_dict[name] = sv
        print("✓")
    except Exception as e:
        print(f"⚠️ {e}")

print(f"\n✓ SHAP computed for {len(shap_values_dict)} feature-based models")

In [ ]:
# ============================================================
# 10.3.2b  SHAP FOR THE DEEP-LEARNING MODELS (KernelExplainer)
# ============================================================
# For deep learning, KernelExplainer is applied to a wrapper that maps a single
# feature vector to a full input window

N_DL_SHAP_BG = 50      # Reduced background, for computational cost
N_DL_SHAP_EXPLAIN = 30  # Samples to explain

for arch_name, r in dl_hpo_results.items():
    print(f"SHAP for {arch_name} (DL)...", end=" ")
    model = r['model']
    model.eval()

    try:
        # Wrapper: receives the flat features of one time step and builds a window from them
        def dl_predict_wrapper(X_flat):
            """Predict from an artificial window built by repeating one feature vector."""
            X_sc = scaler_X.transform(X_flat)
            # Build the window: repeat the same vector WINDOW_SIZE times (approximation)
            X_seq = np.tile(X_sc[:, np.newaxis, :], (1, WINDOW_SIZE, 1))
            X_tensor = torch.tensor(X_seq, dtype=torch.float32).to(device)
            with torch.no_grad():
                preds_sc = model(X_tensor).squeeze().cpu().numpy()
            return scaler_y.inverse_transform(preds_sc.reshape(-1, 1)).flatten()

        bg_sample = shap_background.values[:N_DL_SHAP_BG]
        explain_sample = shap_explain.values[:N_DL_SHAP_EXPLAIN]

        explainer = shap.KernelExplainer(dl_predict_wrapper, bg_sample)
        sv = explainer.shap_values(explain_sample, nsamples=100)
        shap_values_dict[arch_name] = sv
        print("✓")
    except Exception as e:
        print(f"⚠️ {e}")

print(f"\n✓ SHAP total: {len(shap_values_dict)} models")

In [ ]:
# ============================================================
# 10.3.2c  SHAP VISUALIZATION — comparative summary plots
# ============================================================
# Select representative models (one per family; the MoE has no SHAP values)
representative_models = []
for family_target in ['Statistical', 'Machine Learning', 'Deep Learning']:
    for name in shap_values_dict:
        full_name = f'{name} (Opt)' if name in hpo_results else f'{name} (Opt)'
        if full_name in final_results and final_results[full_name]['family'] == family_target:
            representative_models.append(name)
            break
        elif name in final_results and final_results[name]['family'] == family_target:
            representative_models.append(name)
            break

n_rep = len(representative_models)
if n_rep > 0:
    # Number of subplot columns: at most 3 per row, for readability
    ncols = min(n_rep, 3)
    nrows = (n_rep + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 8 * nrows))

    # Make 'axes' a 1-D iterable in every case
    if nrows == 1 and ncols > 1:  # one row with several columns
        axes = axes
    elif nrows == 1 and ncols == 1:  # a single plot
        axes = [axes]
    else:  # several rows
        axes = axes.flatten()

    for ax_idx, name in enumerate(representative_models):
        ax = axes[ax_idx]
        sv = shap_values_dict[name]
        # Make sure the array has the right dimensions
        if isinstance(sv, list):
            sv = sv[0] if len(sv) > 0 else np.zeros((1, len(feature_cols)))  # fallback for empty SHAP values
        n_samples_sv = sv.shape[0]
        explain_data = shap_explain.iloc[:n_samples_sv] if n_samples_sv <= len(shap_explain) else shap_explain

        plt.sca(ax)
        shap.summary_plot(sv, explain_data, plot_type='bar',
                          max_display=15, show=False, plot_size=None)
        ax.set_title(f'{name}', fontweight='bold', fontsize=12)
        ax.set_xlabel('mean |SHAP value|')
        ax.ticklabel_format(style='sci', axis='x', scilimits=(0,0))  # force scientific notation

    # Remove empty axes if there are more subplots than models
    for i in range(ax_idx + 1, len(axes)):
        fig.delaxes(axes[i])

    fig.suptitle('SHAP Feature Importance — Family Representatives',
                 fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(BENCH_DIR / 'fig_shap_summary_comparison.png', bbox_inches='tight')
    plt.show()

# --- SHAP beeswarm of the model in row 2 of the test ranking (summary_df.iloc[1]) ---
best_model_name = summary_df.iloc[1]['Model'].split(' (')[0]
if best_model_name in shap_values_dict:
    sv_best = shap_values_dict[best_model_name]
    if isinstance(sv_best, list):
        sv_best = sv_best[0]
    n_sv = sv_best.shape[0]
    fig, ax = plt.subplots(figsize=(10, 8))
    plt.sca(ax)
    shap.summary_plot(sv_best, shap_explain.iloc[:n_sv],
                      max_display=20, show=False, plot_size=None)
    ax.set_title(f'SHAP Beeswarm — {best_model_name} (Best Overall Model)',
                 fontweight='bold', fontsize=13)
    ax.ticklabel_format(style='sci', axis='x', scilimits=(0,0))  # force scientific notation
    plt.tight_layout()
    plt.savefig(BENCH_DIR / 'fig_shap_beeswarm_best.png', bbox_inches='tight')
    plt.show()

print("✓ SHAP figures saved")

In [ ]:
# ============================================================
# 10.3.2d  INTER-MODEL CONSISTENCY OF THE SHAP RANKINGS
# ============================================================
# Do the models agree on which features are important?
# Spearman correlation between the SHAP importance rankings of the models

shap_importance_rankings = {}
for name, sv in shap_values_dict.items():
    if isinstance(sv, list):
        sv = sv[0]
    # Mean absolute SHAP value per feature
    mean_abs = np.abs(sv).mean(axis=0)
    if len(mean_abs) == len(feature_cols):
        ranking = pd.Series(mean_abs, index=feature_cols).rank(ascending=False)
        shap_importance_rankings[name] = ranking

if len(shap_importance_rankings) >= 2:
    ranking_df = pd.DataFrame(shap_importance_rankings)
    corr_matrix = ranking_df.corr(method='spearman')

    fig, ax = plt.subplots(figsize=(10, 8))
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
    sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
                cmap='RdYlGn', center=0.5, vmin=-1, vmax=1,
                ax=ax, linewidths=0.5, square=True,
                cbar_kws={'label': 'Spearman ρ (SHAP ranking)'})
    ax.set_title('Inter-Model Consistency: SHAP Ranking Correlation',
                 fontweight='bold', fontsize=13)
    plt.tight_layout()
    plt.savefig(BENCH_DIR / 'fig_shap_ranking_correlation.png', bbox_inches='tight')
    plt.show()

    print("\nMean SHAP ranking correlation per model:")
    mean_corr = corr_matrix.mean(axis=1).sort_values(ascending=False)
    for name, val in mean_corr.items():
        print(f"  {name:20s}: ρ̄ = {val:.3f}")
else:
    print("⚠️ Not enough models with SHAP values to compare rankings")

#### 10.3.3 MoE routing — t-SNE/UMAP of the expert assignment

`predict_with_metadata` returns, for every test hour, the expert selected by the gating, the gating mode (`hard`, `soft` or `hard_fallback`) and the minimum classifier confidence `min_conf`. A random sample of up to 5,000 test hours is projected to 2-D with t-SNE (perplexity 30) and, if the `umap` package is installed, UMAP (15 neighbours), after standardizing the features. Each point is coloured by its assigned expert and, in a third panel, by gating mode. This shows **which regions of the input space each expert covers** and whether the specialization is consistent in feature space.

The second cell crosses the expert assignment with the operating context (hour of day and day of week) and plots the distribution of the gating confidence by gating mode, together with the hard-gating threshold of the model.

In [ ]:
# ============================================================
# 10.3.3a  t-SNE / UMAP OF THE EXPERT ASSIGNMENT
# ============================================================
from sklearn.manifold import TSNE

# Use UMAP if it is installed
try:
    import umap
    HAS_UMAP = True
except:
    HAS_UMAP = False
    print("⚠️ UMAP not available; only t-SNE will be used")

if moe_path.exists() and hasattr(moe_model, 'predict_with_metadata'):
    print("Getting MoE routing metadata on the test set...")
    moe_preds_meta, moe_metadata = moe_model.predict_with_metadata(df_test)

    print(f"  Observations: {len(moe_metadata)}")
    print(f"  Gating modes: {moe_metadata['gating_mode'].value_counts().to_dict()}")
    print(f"  Active experts: {moe_metadata['expert_id'].nunique()}")

    # Subsample for t-SNE (at most 5000 points)
    N_TSNE = min(5000, len(moe_metadata))
    idx_sample = np.random.choice(len(moe_metadata), N_TSNE, replace=False)
    X_tsne_input = X_test.iloc[idx_sample].values
    meta_sample = moe_metadata.iloc[idx_sample].reset_index(drop=True)

    # Standardize for t-SNE
    X_tsne_scaled = StandardScaler().fit_transform(X_tsne_input)

    # --- t-SNE ---
    print("\nComputing t-SNE (perplexity=30)...")
    tsne = TSNE(n_components=2, perplexity=30, random_state=RANDOM_SEED,
                n_iter=1000, learning_rate='auto', init='pca')
    X_2d_tsne = tsne.fit_transform(X_tsne_scaled)

    # --- UMAP ---
    if HAS_UMAP:
        print("Computing UMAP (n_neighbors=15)...")
        reducer = umap.UMAP(n_components=2, n_neighbors=15,
                            min_dist=0.1, random_state=RANDOM_SEED)
        X_2d_umap = reducer.fit_transform(X_tsne_scaled)

    # --- Figure ---
    n_plots = 2 if HAS_UMAP else 1
    fig, axes = plt.subplots(1, n_plots + 1, figsize=(7 * (n_plots + 1), 6))

    # Colours by expert
    expert_ids_unique = meta_sample['expert_id'].unique()
    expert_cmap = plt.cm.get_cmap('tab10', len(expert_ids_unique))
    expert_color_map = {eid: expert_cmap(i) for i, eid in enumerate(sorted(expert_ids_unique))}
    colors_expert = [expert_color_map[eid] for eid in meta_sample['expert_id']]

    # t-SNE coloured by expert
    ax = axes[0]
    for eid in sorted(expert_ids_unique):
        mask = meta_sample['expert_id'] == eid
        ax.scatter(X_2d_tsne[mask, 0], X_2d_tsne[mask, 1],
                   c=[expert_color_map[eid]], s=8, alpha=0.6, label=eid)
    ax.set_title('t-SNE — Expert Assignment', fontweight='bold')
    ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')
    ax.legend(fontsize=7, markerscale=3, title='Expert', loc='best')

    # UMAP coloured by expert
    if HAS_UMAP:
        ax = axes[1]
        for eid in sorted(expert_ids_unique):
            mask = meta_sample['expert_id'] == eid
            ax.scatter(X_2d_umap[mask, 0], X_2d_umap[mask, 1],
                       c=[expert_color_map[eid]], s=8, alpha=0.6, label=eid)
        ax.set_title('UMAP — Expert Assignment', fontweight='bold')
        ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')
        ax.legend(fontsize=7, markerscale=3, title='Expert', loc='best')

    # t-SNE coloured by gating mode (hard vs. soft)
    ax = axes[-1]
    mode_colors = {'hard': '#2A9D8F', 'soft': '#E76F51', 'hard_fallback': '#E9C46A'}
    for mode, color in mode_colors.items():
        mask = meta_sample['gating_mode'] == mode
        if mask.sum() > 0:
            ax.scatter(X_2d_tsne[mask, 0], X_2d_tsne[mask, 1],
                       c=color, s=8, alpha=0.6, label=f'{mode} ({mask.sum():,})')
    ax.set_title('t-SNE — Gating Mode', fontweight='bold')
    ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')
    ax.legend(fontsize=9, markerscale=3)

    fig.suptitle('Visualization of MoE Routing in Feature Space',
                 fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.savefig(BENCH_DIR / 'fig_routing_tsne_umap.png', bbox_inches='tight')
    plt.show()

    print("✓ t-SNE/UMAP figures saved")
else:
    print("⚠️ MoE model not available or does not support predict_with_metadata")

In [ ]:
# ============================================================
# 10.3.3b  CONDITIONAL ANALYSIS: EXPERT × OPERATING CONTEXT
# ============================================================
# Which expert is activated in each operating context?
# Cross-tabulation: assigned expert × hour of day and day of week

if moe_path.exists() and 'moe_metadata' in dir():
    # Add the time attributes of the test set
    analysis_df = moe_metadata.copy()
    analysis_df['hour'] = df_test.index[:len(analysis_df)].hour
    analysis_df['dayofweek'] = df_test.index[:len(analysis_df)].dayofweek
    analysis_df['confidence'] = analysis_df['min_conf']

    fig, axes = plt.subplots(1, 3, figsize=(20, 6))

    # 1. Expert distribution by hour of day
    ax = axes[0]
    expert_hour = pd.crosstab(analysis_df['hour'], analysis_df['expert_id'], normalize='index')
    expert_hour.plot(kind='bar', stacked=True, ax=ax, colormap='tab10', width=0.85)
    ax.set_title('Expert Activation by Hour', fontweight='bold')
    ax.set_xlabel('Hour of Day')
    ax.set_ylabel('Proportion')
    ax.legend(title='Expert', fontsize=7, bbox_to_anchor=(1.02, 1))

    # 2. Expert distribution by day of week
    ax = axes[1]
    expert_dow = pd.crosstab(analysis_df['dayofweek'], analysis_df['expert_id'], normalize='index')
    dow_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
    expert_dow.index = [dow_labels[i] for i in expert_dow.index]
    expert_dow.plot(kind='bar', stacked=True, ax=ax, colormap='tab10', width=0.85)
    ax.set_title('Expert Activation by Day', fontweight='bold')
    ax.set_xlabel('Day of Week')
    ax.set_ylabel('Proportion')
    ax.legend(title='Expert', fontsize=7, bbox_to_anchor=(1.02, 1))

    # 3. Distribution of the gating confidence
    ax = axes[2]
    for mode in ['hard', 'soft', 'hard_fallback']:
        mask = analysis_df['gating_mode'] == mode
        if mask.sum() > 0:
            ax.hist(analysis_df.loc[mask, 'confidence'], bins=30, alpha=0.6,
                    label=f'{mode} (n={mask.sum():,})', density=True)
    ax.axvline(x=moe_model.gating_threshold, color='red', linestyle='--',
               linewidth=2, label=f'Threshold={moe_model.gating_threshold}')
    ax.set_title('Gating Confidence Distribution', fontweight='bold')
    ax.set_xlabel('Minimum Confidence (min_conf)')
    ax.set_ylabel('Density')
    ax.legend()

    plt.tight_layout()
    plt.savefig(BENCH_DIR / 'fig_expert_activation_analysis.png', bbox_inches='tight')
    plt.show()

    # Summary table
    print("\n" + "="*80)
    print("  CONDITIONAL ANALYSIS: EXPERT × OPERATING CONTEXT")
    print("="*80)
    print("\nDominant expert by time band:")
    for hour_range, label in [(range(0,6), 'Night (0-5)'),
                               (range(6,12), 'Morning (6-11)'),
                               (range(12,18), 'Afternoon (12-17)'),
                               (range(18,24), 'Evening (18-23)')]:
        mask = analysis_df['hour'].isin(hour_range)
        dominant = analysis_df.loc[mask, 'expert_id'].mode()
        pct = (analysis_df.loc[mask, 'expert_id'] == dominant.iloc[0]).mean() * 100 if len(dominant) > 0 else 0
        print(f"  {label:20s} → {dominant.iloc[0] if len(dominant) > 0 else 'N/A'} ({pct:.1f}%)")

    print("\nDominant expert by day:")
    for dow, label in enumerate(dow_labels):
        mask = analysis_df['dayofweek'] == dow
        dominant = analysis_df.loc[mask, 'expert_id'].mode()
        pct = (analysis_df.loc[mask, 'expert_id'] == dominant.iloc[0]).mean() * 100 if len(dominant) > 0 else 0
        print(f"  {label:5s} → {dominant.iloc[0] if len(dominant) > 0 else 'N/A'} ({pct:.1f}%)")

#### 10.3.4 Routing stability

Do similar samples in feature space consistently receive the same expert? High instability (nearby samples routed to different experts) would indicate fragile decision boundaries.

**Metric**: for a random sample of up to 3,000 test hours (standardized features), the $k=10$ nearest neighbours of each sample (Euclidean distance) are found and the fraction of them assigned to the same expert is computed. A threshold of 80% is used as the reference for a stable routing. The summary statistics, overall and per expert, are saved to `routing_stability.json`.

In [ ]:
# ============================================================
# 10.3.4  ROUTING STABILITY
# ============================================================
from sklearn.neighbors import NearestNeighbors

if moe_path.exists() and 'moe_metadata' in dir():
    print("Computing routing stability...")

    # Use a subset for efficiency
    N_STABILITY = min(3000, len(moe_metadata))
    idx_stab = np.random.choice(len(moe_metadata), N_STABILITY, replace=False)
    X_stab = StandardScaler().fit_transform(X_test.iloc[idx_stab].values)
    experts_stab = moe_metadata.iloc[idx_stab]['expert_id'].values

    # k-NN
    K_NEIGHBORS = 10
    nn_model = NearestNeighbors(n_neighbors=K_NEIGHBORS + 1, metric='euclidean')
    nn_model.fit(X_stab)
    distances, indices = nn_model.kneighbors(X_stab)

    # For each sample, fraction of neighbours assigned to the same expert
    stability_scores = []
    for i in range(N_STABILITY):
        neighbors = indices[i, 1:]  # exclude the sample itself
        same_expert = np.mean(experts_stab[neighbors] == experts_stab[i])
        stability_scores.append(same_expert)

    stability_scores = np.array(stability_scores)

    print(f"\n  Routing stability (k={K_NEIGHBORS} neighbours):")
    print(f"    Mean:    {stability_scores.mean():.4f} ({stability_scores.mean()*100:.1f}%)")
    print(f"    Median:  {np.median(stability_scores):.4f}")
    print(f"    Std:     {stability_scores.std():.4f}")
    print(f"    Min:     {stability_scores.min():.4f}")
    print(f"    >80%:    {(stability_scores > 0.8).mean()*100:.1f}% of the samples")
    print(f"    >90%:    {(stability_scores > 0.9).mean()*100:.1f}% of the samples")
    print(f"    =100%:   {(stability_scores == 1.0).mean()*100:.1f}% of the samples")

    # Figure
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # 1. Stability histogram
    ax = axes[0]
    ax.hist(stability_scores, bins=K_NEIGHBORS + 1, edgecolor='black', alpha=0.7, color='#2A9D8F')
    ax.axvline(x=0.8, color='red', linestyle='--', linewidth=2, label='80% Threshold')
    ax.axvline(x=stability_scores.mean(), color='orange', linestyle='-',
               linewidth=2, label=f'Mean={stability_scores.mean():.2f}')
    ax.set_xlabel(f'Fraction of neighbours (k={K_NEIGHBORS}) with the same expert')
    ax.set_ylabel('Frequency')
    ax.set_title('Routing Stability Distribution', fontweight='bold')
    ax.legend()

    # 2. Stability by expert
    ax = axes[1]
    stab_df = pd.DataFrame({
        'expert_id': experts_stab,
        'stability': stability_scores
    })
    expert_stab = stab_df.groupby('expert_id')['stability'].agg(['mean', 'std', 'count'])
    expert_stab = expert_stab.sort_values('mean', ascending=True)
    bars = ax.barh(expert_stab.index, expert_stab['mean'],
                   xerr=expert_stab['std'], color='#457B9D',
                   edgecolor='white', capsize=3)
    ax.axvline(x=0.8, color='red', linestyle='--', alpha=0.7)
    ax.set_xlabel('Mean Stability')
    ax.set_title('Stability by Expert', fontweight='bold')

    xmax = (expert_stab['mean'] + expert_stab['std']).max()
    ax.set_xlim(0, xmax + 0.20)

    for bar, (_, row) in zip(bars, expert_stab.iterrows()):
        ax.text(row['mean'] + row['std'] + 0.01, bar.get_y() + bar.get_height()/2,
                f'{row["mean"]:.2f} (n={int(row["count"])})', va='center', fontsize=8)

    # 3. t-SNE coloured by stability
    ax = axes[2]
    # Reuse the t-SNE projection of Section 10.3.3 if it exists
    stab_for_tsne = stability_scores[:min(N_STABILITY, N_TSNE)] if 'X_2d_tsne' in dir() else stability_scores
    if 'X_2d_tsne' in dir() and len(X_2d_tsne) >= len(stab_for_tsne):
        sc = ax.scatter(X_2d_tsne[:len(stab_for_tsne), 0],
                        X_2d_tsne[:len(stab_for_tsne), 1],
                        c=stab_for_tsne, cmap='RdYlGn', s=5, alpha=0.6)
        plt.colorbar(sc, ax=ax, label='Stability')
        ax.set_title('t-SNE — Routing Stability', fontweight='bold')
        ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')
    else:
        # Otherwise compute a local t-SNE
        tsne_stab = TSNE(n_components=2, perplexity=30, random_state=RANDOM_SEED).fit_transform(
            X_stab[:min(2000, len(X_stab))])
        sc = ax.scatter(tsne_stab[:, 0], tsne_stab[:, 1],
                        c=stability_scores[:len(tsne_stab)], cmap='RdYlGn', s=5, alpha=0.6)
        plt.colorbar(sc, ax=ax, label='Stability')
        ax.set_title('t-SNE — Routing Stability', fontweight='bold')

    fig.suptitle(f'MoE Routing Stability Analysis (k={K_NEIGHBORS})',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(BENCH_DIR / 'fig_routing_stability.png', bbox_inches='tight')
    plt.show()

    # Save the metrics
    stability_summary = {
        'k_neighbors': K_NEIGHBORS,
        'mean_stability': float(stability_scores.mean()),
        'median_stability': float(np.median(stability_scores)),
        'std_stability': float(stability_scores.std()),
        'pct_above_80': float((stability_scores > 0.8).mean()),
        'pct_above_90': float((stability_scores > 0.9).mean()),
        'pct_perfect': float((stability_scores == 1.0).mean()),
        'per_expert': expert_stab['mean'].to_dict(),
    }
    with open(BENCH_DIR / 'routing_stability.json', 'w') as f:
        json.dump(stability_summary, f, indent=2, default=str)

    print("\n✓ Stability analysis saved")
else:
    print("⚠️ MoE model not available for the stability analysis")

#### 10.3.5 Consolidated interpretability summary

Summary of Sections 10.3.1–10.3.4: interpretability score of every model, the five features with the highest mean |SHAP| averaged over the baselines, the share of hard and soft gating decisions of the MoE on the test set, and its routing stability. The closing statement printed by the cell is a fixed text written by the authors, not a computed result.

In [ ]:
# ============================================================
# 10.3.5  CONSOLIDATED INTERPRETABILITY SUMMARY
# ============================================================
print("\n" + "=" * 100)
print("  CONSOLIDATED SUMMARY — INTERPRETABILITY ANALYSIS")
print("=" * 100)

print("\n  1. QUANTITATIVE SCALE (1-5):")
for _, row in interp_score_df.iterrows():
    print(f"     {row['Model']:30s} → {int(row['Score (1-5)'])}/5  ({row['Justification']})")

print("\n  2. SHAP VALUES:")
print(f"     Models analysed: {len(shap_values_dict)}")
if shap_values_dict:
    # Most important features (mean |SHAP| averaged across models)
    all_importances = {}
    for name, sv in shap_values_dict.items():
        if isinstance(sv, list): sv = sv[0]
        if sv.shape[1] == len(feature_cols):
            for i, feat in enumerate(feature_cols):
                if feat not in all_importances:
                    all_importances[feat] = []
                all_importances[feat].append(np.abs(sv[:, i]).mean())
    if all_importances:
        avg_imp = {k: np.mean(v) for k, v in all_importances.items()}
        top5_global = sorted(avg_imp.items(), key=lambda x: -x[1])[:5]
        print("     Top-5 most important features (averaged across models):")
        for feat, imp in top5_global:
            print(f"       {feat:30s}: {imp:.4f}")

if moe_path.exists() and 'stability_scores' in dir():
    print(f"\n  3. ROUTING MoE:")
    print(f"     Active experts: {moe_metadata['expert_id'].nunique()}")
    print(f"     Hard gating: {(moe_metadata['gating_mode']=='hard').mean()*100:.1f}%")
    print(f"     Soft gating: {(moe_metadata['gating_mode']=='soft').mean()*100:.1f}%")
    print(f"\n  4. ROUTING STABILITY:")
    print(f"     Mean: {stability_scores.mean()*100:.1f}%")
    print(f"     Samples with >80% consistency: {(stability_scores > 0.8).mean()*100:.1f}%")

print("\n  5. CONCLUSION:")
print("     The MoE offers a distinctive balance: operational interpretability (score 4/5)")
print("     thanks to the explicit gating by day type / week type, combined with the")
print("     predictive capacity of domain-specialized models.")
print("=" * 100)

# Save
interp_score_df.to_csv(BENCH_DIR / 'interpretability_scores.csv', index=False)
print(f"\n✓ All interpretability files saved to {BENCH_DIR}")

### 10.4 Statistical tests

Whether the differences between models are statistically significant is tested on the **absolute error of every test observation** of each model (MoE and tuned baselines):

1. **One-way ANOVA** — $H_0$: all models have the same mean absolute error. Its assumptions are checked with the Shapiro-Wilk test (normality, on a random subsample of 500 errors per model) and Levene's test (homogeneity of variances).
2. **Friedman test** — non-parametric alternative for paired samples (the models are evaluated on the same test hours).
3. **Tukey HSD** — parametric post-hoc pairwise comparisons, run if the ANOVA is significant ($\alpha = 0.05$).
4. **Pairwise Wilcoxon signed-rank tests with Bonferroni correction** — non-parametric post-hoc comparisons, run if the Friedman test is significant. They are used in place of the Nemenyi test.

The error vectors of all models are truncated to the length of the shortest one (the deep-learning models have no prediction for the first 168 test hours) so that the paired tests receive vectors of equal length.

> Reference: Demšar, J. (2006). *Statistical Comparisons of Classifiers over Multiple Data Sets*. Journal of Machine Learning Research, 7, 1–30.

In [ ]:
# ============================================================
# 10.4a  PER-OBSERVATION ERRORS
# ============================================================
# The statistical tests need the individual error of every time step
errors_dict = {}

# MoE
if moe_preds_test is not None:
    errors_dict['MoE'] = np.abs(y_test.values - moe_preds_test)

# Feature-based models
for name, r in hpo_results.items():
    if r['type'] == 'feature_based':
        model = r['model']
        preds = model.predict(X_test)
        errors_dict[name] = np.abs(y_test.values - preds)

# Deep-learning models
for name, r in dl_hpo_results.items():
    model = r['model']
    X_test_sc_dl = scaler_X.transform(X_test)
    y_test_sc_dl = scaler_y.transform(y_test.values.reshape(-1,1)).flatten()
    test_ds_dl = TimeSeriesDataset(X_test_sc_dl, y_test_sc_dl, WINDOW_SIZE)
    test_ld = DataLoader(test_ds_dl, batch_size=256, shuffle=False)
    preds = predict_dl(model, test_ld, scaler_y)
    # Align lengths
    n_preds = min(len(preds), len(y_test) - WINDOW_SIZE)
    errors_dict[name] = np.abs(y_test.values[WINDOW_SIZE:WINDOW_SIZE+n_preds] - preds[:n_preds])

# Truncate all error vectors to the shortest length
min_len = min(len(e) for e in errors_dict.values()) if errors_dict else 0
for k in errors_dict:
    errors_dict[k] = errors_dict[k][:min_len]

errors_df = pd.DataFrame(errors_dict)
print(f"Absolute errors computed: {errors_df.shape[0]} observations × {errors_df.shape[1]} models")
print(f"Models: {list(errors_dict.keys())}")

In [ ]:
# ============================================================
# 10.4b  ONE-WAY ANOVA
# ============================================================
print("="*80)
print("  ONE-WAY ANOVA")
print("="*80)
print("  H₀: the mean errors of all models are equal")
print("  H₁: at least one model has a different mean error\n")

groups = [errors_dict[k] for k in errors_dict]
f_stat, p_value_anova = sp_stats.f_oneway(*groups)

print(f"  F-statistic: {f_stat:.4f}")
print(f"  p-value:     {p_value_anova:.2e}")
print(f"  Significant (α=0.05): {'✓ YES' if p_value_anova < 0.05 else '✗ NO'}")
print(f"  Significant (α=0.01): {'✓ YES' if p_value_anova < 0.01 else '✗ NO'}")

# Check the assumptions
# Normality (Shapiro-Wilk on subsamples)
print("\n  Assumption checks:")
for name, errors in errors_dict.items():
    sample = np.random.choice(errors, min(500, len(errors)), replace=False)
    _, p_shapiro = sp_stats.shapiro(sample)
    print(f"    {name:20s} — Shapiro-Wilk p={p_shapiro:.4f} "
          f"({'Normal' if p_shapiro > 0.05 else 'Not normal'})")

# Levene (homogeneity of variances)
_, p_levene = sp_stats.levene(*groups)
print(f"\n    Levene (homoscedasticity): p={p_levene:.4f} "
      f"({'Homogeneous' if p_levene > 0.05 else 'Heterogeneous'})")

In [ ]:
# ============================================================
# 10.4c  FRIEDMAN TEST (non-parametric)
# ============================================================
print("\n" + "="*80)
print("  FRIEDMAN TEST (non-parametric alternative)")
print("="*80)
print("  H₀: there is no difference in the ranking of the models across observations")
print("  H₁: at least one model has a different ranking\n")

# Friedman requires samples of equal size (already aligned)
try:
    chi2_stat, p_value_friedman = sp_stats.friedmanchisquare(*groups)
    print(f"  χ² statistic: {chi2_stat:.4f}")
    print(f"  p-value:      {p_value_friedman:.2e}")
    print(f"  Significant (α=0.05): {'✓ YES' if p_value_friedman < 0.05 else '✗ NO'}")
except Exception as e:
    print(f"  ⚠️ Friedman test failed: {e}")
    p_value_friedman = 1.0

In [ ]:
# ============================================================
# 10.4d  POST-HOC TEST — TUKEY HSD
# ============================================================
from itertools import combinations

print("\n" + "="*80)
print("  POST-HOC TESTS — Tukey HSD (pairwise comparisons)")
print("="*80)

if p_value_anova < 0.05:
    print("  ANOVA significant → pairwise comparisons\n")

    # Tukey HSD with statsmodels
    from statsmodels.stats.multicomp import pairwise_tukeyhsd

    all_errors = []
    all_labels = []
    for name, errors in errors_dict.items():
        all_errors.extend(errors)
        all_labels.extend([name] * len(errors))

    tukey = pairwise_tukeyhsd(all_errors, all_labels, alpha=0.05)
    print(tukey.summary())

    # Save the results
    tukey_df = pd.DataFrame(data=tukey._results_table.data[1:],
                            columns=tukey._results_table.data[0])
    tukey_df.to_csv(BENCH_DIR / 'tukey_hsd_results.csv', index=False)

    # Simultaneous confidence intervals
    fig, ax = plt.subplots(figsize=(14, 8))
    tukey.plot_simultaneous(ax=ax)
    ax.set_title('Tukey HSD — Simultaneous Confidence Intervals', fontweight='bold')
    ax.set_xlabel('Mean Absolute Error')
    plt.tight_layout()
    plt.savefig(BENCH_DIR / 'fig_tukey_hsd.png', bbox_inches='tight')
    plt.show()
else:
    print("  ANOVA not significant → no post-hoc tests required")

In [ ]:
# ============================================================
# 10.4e  NON-PARAMETRIC POST-HOC (pairwise Wilcoxon, Bonferroni-corrected)
# ============================================================
print("\n" + "="*80)
print("  NON-PARAMETRIC POST-HOC — pairwise Wilcoxon signed-rank tests")
print("="*80)

if p_value_friedman < 0.05:
    print("  Friedman significant → pairwise Wilcoxon comparisons\n")

    model_names = list(errors_dict.keys())
    n_models = len(model_names)
    pairwise_pvalues = np.ones((n_models, n_models))

    for i, j in combinations(range(n_models), 2):
        _, p_val = sp_stats.wilcoxon(
            errors_dict[model_names[i]],
            errors_dict[model_names[j]],
            alternative='two-sided'
        )
        # Bonferroni correction
        n_comparisons = n_models * (n_models - 1) / 2
        p_corrected = min(p_val * n_comparisons, 1.0)
        pairwise_pvalues[i, j] = p_corrected
        pairwise_pvalues[j, i] = p_corrected

    # Heatmap of the p-values
    pairwise_df = pd.DataFrame(pairwise_pvalues, index=model_names, columns=model_names)

    fig, ax = plt.subplots(figsize=(12, 10))
    mask = np.triu(np.ones_like(pairwise_pvalues, dtype=bool))
    sns.heatmap(pairwise_df, mask=mask, annot=True, fmt='.4f',
                cmap='RdYlGn', center=0.05, vmin=0, vmax=1,
                ax=ax, linewidths=0.5,
                cbar_kws={'label': 'corrected p-value (Bonferroni)'})
    ax.set_title('Pairwise Wilcoxon Test (Bonferroni-corrected)', fontweight='bold')
    plt.tight_layout()
    plt.savefig(BENCH_DIR / 'fig_pairwise_wilcoxon.png', bbox_inches='tight')
    plt.show()

    pairwise_df.to_csv(BENCH_DIR / 'pairwise_wilcoxon_pvalues.csv')
    print("✓ Results saved")
else:
    print("  Friedman not significant → no post-hoc comparisons required")

Boxplots of the per-observation absolute errors used in the tests above (outliers hidden), sorted by median error and coloured by model family.

In [ ]:
# ============================================================
# 10.4f  ERROR DISTRIBUTIONS — COMPARATIVE BOXPLOT
# ============================================================
fig, ax = plt.subplots(figsize=(16, 8))

# Sort the models by median error
medians = errors_df.median().sort_values()
errors_ordered = errors_df[medians.index]

bp = ax.boxplot([errors_ordered[col].values for col in errors_ordered.columns],
                labels=errors_ordered.columns,
                patch_artist=True, showfliers=False,
                medianprops=dict(color='black', linewidth=2))

# Colour by family
for i, col in enumerate(errors_ordered.columns):
    family = final_results.get(f'{col} (Opt)', final_results.get(f'{col} (Opt)',
             final_results.get(f'{col} (Prop)', {})))
    fam = family.get('family', 'Unknown') if isinstance(family, dict) else 'Unknown'
    bp['boxes'][i].set_facecolor(family_colors.get(fam, '#CCCCCC'))
    bp['boxes'][i].set_alpha(0.7)

ax.set_ylabel('Absolute Error (kWh)')
ax.set_title('Distribution of Absolute Errors by Model', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')

patches = [mpatches.Patch(color=c, label=f, alpha=0.7) for f, c in family_colors.items()]
ax.legend(handles=patches, loc='upper right')
plt.tight_layout()
plt.savefig(BENCH_DIR / 'fig_error_distributions.png', bbox_inches='tight')
plt.show()

---
### 10.5 Robustness test: atypical COVID-19 behaviour (2020–2021)

The years 2020 and 2021 were **deliberately excluded from the training data** of the MoE and of every benchmark model because of the disruptive impact of COVID-19 on electricity demand patterns. According to Buechler et al. (2020), the pandemic caused drops of up to 26% in daily demand in some countries, producing unprecedented behaviour in the time series.

This analysis uses those excluded data as an **out-of-distribution (OOD) stress test**: it evaluates how models trained on "normal" data (2022–2025) behave when they face patterns they have never seen. The compared models are the MoE and the best tuned model of each baseline family (the models selected in Section 7). Each model is applied to each period without re-training; the demand-lag features are taken from the actual 2020–2021 series.

#### Critical periods selected for Colombia

| # | Period | Event | Expected impact on demand |
|---|---------|--------|-----------------------------|
| 1 | **Jan 6 – Mar 15, 2020** | Pre-COVID (normal baseline) | Reference: typical pattern before the shock |
| 2 | **Mar 25 – Apr 30, 2020** | Strict national lockdown (Decree 457) | Massive drop: industrial and commercial closures, mobility restrictions |
| 3 | **Apr 4 – Apr 12, 2020** | Holy Week under full lockdown | Historic minimum demand: holiday and lockdown at the same time |
| 4 | **Jun 1 – Aug 31, 2020** | Gradual, selective reopening | Oscillating recovery: sectors reopen at different paces |
| 5 | **Dec 15, 2020 – Jan 15, 2021** | Christmas season during the pandemic | Atypical holidays: partial restrictions plus a new wave |
| 6 | **Apr 28 – May 31, 2021** | National strike (social unrest) | Economic disruption: road blockades, sector-specific curfews |

> **Note**: none of these data were used to train, validate or test the models in Sections 4–10.4.

In [ ]:
# ============================================================
# 10.5.0  LOAD THE FULL DATA (INCLUDING 2020-2021)
# ============================================================
# features.parquet contains the WHOLE history since 2020;
# the 2020-2021 rows are extracted here

print("=" * 80)
print("  LOADING COVID DATA (2020-2021) — excluded from training")
print("=" * 80)

# Reload the full dataset
features_path_full = Path(DATA_DIR) / 'processed' / 'features.parquet'

if features_path_full.exists():
    df_full = pd.read_parquet(features_path_full)
    if not isinstance(df_full.index, pd.DatetimeIndex):
        for col in ['timestamp', 'datetime', 'date']:
            if col in df_full.columns:
                df_full = df_full.set_index(col)
                break
    df_full = df_full.sort_index()

    # Keep only 2020-2021
    df_covid_era = df_full.loc['2020-01-01':'2021-12-31'].copy()
    print(f"  2020-2021 data: {len(df_covid_era):,} rows")
    print(f"  Range: {df_covid_era.index.min()} → {df_covid_era.index.max()}")

    # Check that the model features exist
    missing_feats = [c for c in feature_cols if c not in df_covid_era.columns]
    if missing_feats:
        print(f"  ⚠️ Features missing in the COVID data: {missing_feats[:10]}...")
        # Continue with the available features
        available_feats = [c for c in feature_cols if c in df_covid_era.columns]
        print(f"  Available features: {len(available_feats)}/{len(feature_cols)}")
    else:
        available_feats = feature_cols
        print(f"  ✓ All {len(feature_cols)} features available")

    # Drop missing values
    df_covid_clean = df_covid_era[[TARGET] + available_feats].dropna()
    print(f"  Clean rows: {len(df_covid_clean):,}")

    # Demand statistics
    demand_covid = df_covid_clean[TARGET]
    print(f"\n  Demand statistics 2020-2021:")
    print(f"    Mean:   {demand_covid.mean():,.0f} kWh")
    print(f"    Min:    {demand_covid.min():,.0f} kWh")
    print(f"    Max:    {demand_covid.max():,.0f} kWh")
    print(f"    Std:    {demand_covid.std():,.0f} kWh")

    # Compare with the training data (2022-2025)
    demand_train = y_train
    print(f"\n  Comparison with the training data (2022-2025):")
    print(f"    Train mean:  {demand_train.mean():,.0f} kWh")
    print(f"    COVID mean:  {demand_covid.mean():,.0f} kWh")
    pct_diff = (demand_covid.mean() - demand_train.mean()) / demand_train.mean() * 100
    print(f"    Difference:  {pct_diff:+.1f}%")

else:
    print("⚠️ features.parquet not found")
    print("   Generating synthetic COVID data for demonstration...")

    # Synthetic data that mimic the COVID impact
    np.random.seed(RANDOM_SEED)
    idx_covid = pd.date_range('2020-01-01', '2021-12-31', freq='h')
    n_covid = len(idx_covid)

    # Base demand + seasonality
    base = (7000 + 800 * np.sin(2*np.pi*np.arange(n_covid)/24)
            + 300 * np.sin(2*np.pi*np.arange(n_covid)/168)
            + np.random.normal(0, 150, n_covid))

    # Simulated COVID drop (March-May 2020)
    covid_mask = (idx_covid >= '2020-03-25') & (idx_covid <= '2020-05-31')
    base[covid_mask] *= 0.78  # ~22% drop

    # Simulated gradual recovery (June-December 2020)
    recovery_mask = (idx_covid >= '2020-06-01') & (idx_covid <= '2020-12-31')
    recovery_factor = np.linspace(0.82, 0.95, recovery_mask.sum())
    base[recovery_mask] *= recovery_factor

    # Simulated national strike (April-May 2021)
    strike_mask = (idx_covid >= '2021-04-28') & (idx_covid <= '2021-05-31')
    base[strike_mask] *= 0.88

    base = base.clip(4000, 12000)
    df_covid_clean = pd.DataFrame({TARGET: base}, index=idx_covid)
    df_covid_clean.index.name = 'timestamp'

    # Synthetic features
    df_covid_clean['hour'] = df_covid_clean.index.hour
    df_covid_clean['dayofweek'] = df_covid_clean.index.dayofweek
    df_covid_clean['month'] = df_covid_clean.index.month
    df_covid_clean['hour_sin']  = np.sin(2*np.pi*df_covid_clean['hour']/24)
    df_covid_clean['hour_cos']  = np.cos(2*np.pi*df_covid_clean['hour']/24)
    df_covid_clean['dow_sin']   = np.sin(2*np.pi*df_covid_clean['dayofweek']/7)
    df_covid_clean['dow_cos']   = np.cos(2*np.pi*df_covid_clean['dayofweek']/7)
    df_covid_clean['month_sin'] = np.sin(2*np.pi*df_covid_clean['month']/12)
    df_covid_clean['month_cos'] = np.cos(2*np.pi*df_covid_clean['month']/12)
    df_covid_clean['is_holiday'] = 0
    df_covid_clean['is_bridge']  = 0
    for lag in [1,2,3,23,24,25,47,48,49,167,168,169]:
        df_covid_clean[f'demand_lag_{lag}'] = df_covid_clean[TARGET].shift(lag)
    for w in [6,12,24,48,168]:
        df_covid_clean[f'demand_rmean_{w}'] = df_covid_clean[TARGET].shift(1).rolling(w).mean()
        df_covid_clean[f'demand_rstd_{w}']  = df_covid_clean[TARGET].shift(1).rolling(w).std()
    df_covid_clean.dropna(inplace=True)
    available_feats = [c for c in feature_cols if c in df_covid_clean.columns]
    print(f"  Synthetic COVID data: {len(df_covid_clean):,} rows")

#### 10.5.1 Definition of the critical periods

Start and end dates, display label, description, colour and expected behaviour of each period. The cell also reports how many hours of clean data each period contains and its mean demand.

In [ ]:
# ============================================================
# 10.5.1  DEFINITION OF THE CRITICAL PERIODS
# ============================================================

COVID_PERIODS = {
    'P1_PreCOVID': {
        'start': '2020-01-06', 'end': '2020-03-15',
        'label': 'Pre-COVID\n(Baseline)',
        'description': 'Normal pattern before the pandemic shock',
        'color': '#2A9D8F',
        'expected': 'Reference: models should perform reasonably well'
    },
    'P2_Lockdown': {
        'start': '2020-03-25', 'end': '2020-04-30',
        'label': 'Strict\nLockdown',
        'description': 'Decree 457: full national lockdown',
        'color': '#E63946',
        'expected': 'Massive demand drop (~15-25%). Maximum stress for the models'
    },
    'P3_HolyWeek': {
        'start': '2020-04-04', 'end': '2020-04-12',
        'label': 'Holy Week\n+ Lockdown',
        'description': 'Religious holiday during full lockdown',
        'color': '#D62828',
        'expected': 'Historic minimum demand: holiday and lockdown at the same time'
    },
    'P4_Reopening': {
        'start': '2020-06-01', 'end': '2020-08-31',
        'label': 'Gradual\nReopening',
        'description': 'Sectors reopen under protocols; oscillating demand',
        'color': '#F4A261',
        'expected': 'Partial recovery. Models should improve with respect to P2'
    },
    'P5_ChristmasCOVID': {
        'start': '2020-12-15', 'end': '2021-01-15',
        'label': 'Pandemic\nChristmas',
        'description': 'Holiday season with partial restrictions + new wave',
        'color': '#E9C46A',
        'expected': 'Atypical holidays: neither the normal pattern nor full lockdown'
    },
    'P6_NationalStrike': {
        'start': '2021-04-28', 'end': '2021-05-31',
        'label': 'National\nStrike',
        'description': 'Social unrest: road blockades, sector-specific curfews',
        'color': '#264653',
        'expected': 'Localized economic disruption. Irregular demand'
    },
}

# Check data availability
print("Selected critical COVID periods:")
print("-" * 90)
for pid, info in COVID_PERIODS.items():
    mask = (df_covid_clean.index >= info['start']) & (df_covid_clean.index <= info['end'])
    n_hours = mask.sum()
    if n_hours > 0:
        mean_demand = df_covid_clean.loc[mask, TARGET].mean()
        print(f"  {pid:20s} | {info['start']} → {info['end']} | "
              f"{n_hours:5,}h | Mean demand: {mean_demand:,.0f} kWh")
        info['n_hours'] = n_hours
        info['available'] = True
    else:
        print(f"  {pid:20s} | {info['start']} → {info['end']} | ⚠️ NO DATA")
        info['available'] = False

In [ ]:
torch.cuda.empty_cache()

#### 10.5.2 Predictions per period — MoE vs. the best baselines

For each period, the MoE and the best tuned model of each baseline family predict every hour. The MoE receives the full period DataFrame; the feature-based models receive the feature matrix. The deep-learning model needs the previous 168 hours of each period, so its first 168 hours have no prediction (NaN) and are excluded from its metrics. The cell after the predictions prints and saves the metrics per period and model (`covid_stress_test_results.csv`).

In [ ]:
# ============================================================
# 10.5.2  PREDICTIONS IN THE COVID PERIODS
# ============================================================

# Collect the best (tuned) model of each family
covid_models = OrderedDict()

# MoE
if moe_path.exists():
    covid_models['MoE (Proposed)'] = {
        'model': moe_model,
        'family': 'MoE',
        'predict_fn': lambda m, X_df: m.predict(X_df),
        'needs_df': True,  # the MoE needs the full DataFrame
    }

# Best feature-based statistical model
for name in list(top3_stat.keys()):
    if name in hpo_results and hpo_results[name]['type'] == 'feature_based':
        covid_models[f'{name} (Statistical)'] = {
            'model': hpo_results[name]['model'],
            'family': 'Statistical',
            'predict_fn': lambda m, X: m.predict(X),
            'needs_df': False,
        }
        break

# Best ML model
for name in list(top3_ml.keys()):
    if name in hpo_results and hpo_results[name]['type'] == 'feature_based':
        covid_models[f'{name} (Machine Learning)'] = {
            'model': hpo_results[name]['model'],
            'family': 'ML',
            'predict_fn': lambda m, X: m.predict(X),
            'needs_df': False,
        }
        break

# Best DL model
for name in list(top3_dl.keys()):
    if name in dl_hpo_results:
        dl_model_covid = dl_hpo_results[name]['model']
        def dl_predict_covid(model, X_df, scaler_X=scaler_X, scaler_y=scaler_y):
            X_sc = scaler_X.transform(X_df)
            y_dummy = np.zeros(len(X_sc))
            ds = TimeSeriesDataset(X_sc, y_dummy, WINDOW_SIZE)
            loader = DataLoader(ds, batch_size=256, shuffle=False)
            return predict_dl(model, loader, scaler_y)

        covid_models[f'{name} (Deep Learning)'] = {
            'model': dl_model_covid,
            'family': 'Deep Learning',
            'predict_fn': dl_predict_covid,
            'needs_df': False,
            'is_dl': True,
        }
        break

print(f"Models for the COVID evaluation: {list(covid_models.keys())}")

# --- Predictions per period ---
covid_results = {}

for pid, info in COVID_PERIODS.items():
    if not info.get('available', False):
        continue

    mask = (df_covid_clean.index >= info['start']) & (df_covid_clean.index <= info['end'])
    df_period = df_covid_clean.loc[mask]
    X_period = df_period[available_feats]
    y_period = df_period[TARGET]

    period_results = {'y_true': y_period}

    for model_name, minfo in covid_models.items():
        try:
            if model_name.startswith('MoE'):
                # The MoE needs the full DataFrame
                preds = minfo['predict_fn'](minfo['model'], df_period)
            elif minfo.get('is_dl', False):
                # The DL model needs an input window
                preds_raw = minfo['predict_fn'](minfo['model'], X_period)
                # Align lengths (the DL model loses WINDOW_SIZE samples at the start)
                if len(preds_raw) < len(y_period):
                    # Pad with NaN at the start
                    preds = np.full(len(y_period), np.nan)
                    preds[-len(preds_raw):] = preds_raw
                else:
                    preds = preds_raw[:len(y_period)]
            else:
                preds = minfo['predict_fn'](minfo['model'], X_period)

            # Metrics (excluding NaN for the DL model)
            valid_mask = ~np.isnan(preds) if isinstance(preds, np.ndarray) else np.ones(len(preds), dtype=bool)
            if valid_mask.sum() > 0:
                metrics = compute_metrics(y_period.values[valid_mask], preds[valid_mask])
            else:
                metrics = {'MAE': np.nan, 'RMSE': np.nan, 'MAPE': np.nan, 'R2': np.nan}

            period_results[model_name] = {'preds': preds, 'metrics': metrics}
        except Exception as e:
            print(f"  ⚠️ {model_name} failed in {pid}: {e}")
            period_results[model_name] = {
                'preds': np.full(len(y_period), np.nan),
                'metrics': {'MAE': np.nan, 'RMSE': np.nan, 'MAPE': np.nan, 'R2': np.nan}
            }

    covid_results[pid] = period_results

print("\n✓ Predictions generated for all periods")

In [ ]:
# ============================================================
# 10.5.2b  COMPARISON TABLE PER PERIOD
# ============================================================
print("\n" + "=" * 110)
print("  RESULTS — COVID-19 STRESS TEST (data never seen by the models)")
print("=" * 110)

covid_summary_rows = []

for pid, info in COVID_PERIODS.items():
    if pid not in covid_results:
        continue
    results = covid_results[pid]
    print(f"\n  ▸ {pid}: {info['description']}")
    print(f"    Period: {info['start']} → {info['end']} | {info.get('n_hours', 0):,}h")
    print(f"    Expected impact: {info['expected']}")
    print(f"    {'Model':30s} | {'MAPE (%)':>10s} | {'RMSE':>12s} | {'MAE':>12s} | {'R²':>8s}")
    print(f"    {'-'*30}-+-{'-'*10}-+-{'-'*12}-+-{'-'*12}-+-{'-'*8}")

    for model_name in covid_models.keys():
        if model_name in results:
            m = results[model_name]['metrics']
            print(f"    {model_name:30s} | {m['MAPE']:10.4f} | {m['RMSE']:12.2f} | "
                  f"{m['MAE']:12.2f} | {m['R2']:8.4f}")
            covid_summary_rows.append({
                'Period': pid,
                'Description': info['description'],
                'Model': model_name,
                'Family': covid_models[model_name]['family'],
                'MAPE (%)': m['MAPE'],
                'RMSE': m['RMSE'],
                'MAE': m['MAE'],
                'R²': m['R2'],
            })

covid_summary_df = pd.DataFrame(covid_summary_rows)
covid_summary_df.to_csv(BENCH_DIR / 'covid_stress_test_results.csv', index=False)
print(f"\n✓ Results saved to {BENCH_DIR / 'covid_stress_test_results.csv'}")

#### 10.5.3 Time series per critical period

Actual demand (black) and the predictions of each model (dashed) in every period, with the period MAPE of each model in the legend.

In [ ]:
# ============================================================
# 10.5.3  TIME SERIES — PREDICTIONS vs. ACTUAL
# ============================================================
available_periods = [pid for pid, info in COVID_PERIODS.items() if info.get('available', False)]
n_periods = len(available_periods)

fig, axes = plt.subplots(n_periods, 1, figsize=(18, 5 * n_periods), sharex=False)
if n_periods == 1:
    axes = [axes]

model_colors = {}
for i, (name, minfo) in enumerate(covid_models.items()):
    model_colors[name] = family_colors.get(minfo['family'], f'C{i}')

for ax, pid in zip(axes, available_periods):
    info = COVID_PERIODS[pid]
    results = covid_results[pid]
    y_true = results['y_true']

    # Actual demand
    ax.plot(y_true.index, y_true.values, color='black', linewidth=1.5,
            label='Actual', alpha=0.9, zorder=10)

    # Predictions of each model
    for model_name in covid_models.keys():
        if model_name in results:
            preds = results[model_name]['preds']
            mape = results[model_name]['metrics']['MAPE']
            color = model_colors[model_name]
            label_str = f"{model_name} (MAPE: {mape:.2f}%)"
            ax.plot(y_true.index, preds, color=color, linewidth=1.0,
                    alpha=0.75, label=label_str, linestyle='--')

    # Formatting
    ax.set_title(f"{pid}: {info['description']}", fontweight='bold', fontsize=12,
                 color=info['color'])
    ax.set_ylabel('Demand (kWh)')
    ax.legend(loc='upper right', fontsize=8, framealpha=0.9)

    # Shade the period
    ax.axvspan(pd.Timestamp(info['start']), pd.Timestamp(info['end']),
               alpha=0.05, color=info['color'])

fig.suptitle('COVID-19 Stress Test: Predictions vs. Actual Demand\n'
             '(2020-2021 data never seen during training)',
             fontsize=16, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig(BENCH_DIR / 'fig_covid_timeseries.png', bbox_inches='tight')
plt.show()
print("✓ Figure saved: fig_covid_timeseries.png")

#### 10.5.4 Degradation analysis per period

Heatmap of the MAPE of each model in each period, and relative degradation of each model: the mean MAPE over the COVID periods compared with its MAPE on the normal test set of Section 9, $(\text{MAPE}_{\text{COVID}} - \text{MAPE}_{\text{test}}) / \text{MAPE}_{\text{test}} \times 100$. The boxplots show the distribution of the hourly absolute percentage error of each model in each period.

In [ ]:
# ============================================================
# 10.5.4a  DEGRADATION HEATMAP — MAPE PER PERIOD × MODEL
# ============================================================
if len(covid_summary_df) > 0:
    # Pivot: rows = periods, columns = models, values = MAPE
    pivot_mape = covid_summary_df.pivot_table(
        index='Period', columns='Model', values='MAPE (%)', aggfunc='first'
    )

    # Chronological order of the periods
    period_order = [p for p in COVID_PERIODS.keys() if p in pivot_mape.index]
    pivot_mape = pivot_mape.reindex(period_order)

    # Descriptive labels
    period_labels = [f"{pid}\n{COVID_PERIODS[pid]['description'][:35]}"
                     for pid in period_order]

    fig, ax = plt.subplots(figsize=(14, 8))
    sns.heatmap(pivot_mape, annot=True, fmt='.2f', cmap='YlOrRd',
                linewidths=0.5, ax=ax, cbar_kws={'label': 'MAPE (%)'},
                yticklabels=period_labels)
    ax.set_title('Model Degradation in the COVID-19 Periods\n'
                 '(higher values = worse OOD performance)',
                 fontsize=14, fontweight='bold')
    ax.set_xlabel('Model')
    ax.set_ylabel('')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.savefig(BENCH_DIR / 'fig_covid_degradation_heatmap.png', bbox_inches='tight')
    plt.show()

    # --- Relative MAPE: degradation with respect to the normal test set ---
    print("\n" + "=" * 90)
    print("  RELATIVE DEGRADATION — COVID MAPE vs. normal test MAPE")
    print("=" * 90)
    print(f"  {'Model':30s} | {'MAPE Test':>10s} | {'MAPE COVID':>11s} | {'Degradation':>12s}")
    print(f"  {'-'*30}-+-{'-'*10}-+-{'-'*11}-+-{'-'*12}")

    for model_name in covid_models.keys():
        # MAPE on the normal test set
        for fname, r in final_results.items():
            if model_name.split(' (')[0] in fname:
                mape_normal = r['test_metrics']['MAPE']
                break
        else:
            mape_normal = np.nan

        # Mean MAPE over the COVID periods
        model_covid = covid_summary_df[covid_summary_df['Model'] == model_name]
        mape_covid_avg = model_covid['MAPE (%)'].mean()

        if not np.isnan(mape_normal) and mape_normal > 0:
            degradation = ((mape_covid_avg - mape_normal) / mape_normal) * 100
            print(f"  {model_name:30s} | {mape_normal:10.4f} | {mape_covid_avg:11.4f} | "
                  f"{degradation:+11.1f}%")
        else:
            print(f"  {model_name:30s} | {'N/A':>10s} | {mape_covid_avg:11.4f} | {'N/A':>12s}")

In [ ]:
# ============================================================
# 10.5.4b  BOXPLOT — ERROR DISTRIBUTION PER PERIOD
# ============================================================
fig, axes = plt.subplots(1, len(covid_models), figsize=(6 * len(covid_models), 7),
                         sharey=True)
if len(covid_models) == 1:
    axes = [axes]

for ax, (model_name, minfo) in zip(axes, covid_models.items()):
    errors_by_period = []
    period_names = []

    for pid in period_order:
        if pid in covid_results and model_name in covid_results[pid]:
            preds = covid_results[pid][model_name]['preds']
            y_true = covid_results[pid]['y_true'].values
            valid = ~np.isnan(preds)
            if valid.sum() > 0:
                abs_pct_error = np.abs((y_true[valid] - preds[valid]) / y_true[valid]) * 100
                errors_by_period.append(abs_pct_error)
                short_label = COVID_PERIODS[pid]['label'].replace('\n', ' ')
                period_names.append(short_label)

    if errors_by_period:
        bp = ax.boxplot(errors_by_period, labels=period_names, patch_artist=True,
                        showfliers=False, medianprops=dict(color='black', linewidth=2))
        for i, (patch, pid) in enumerate(zip(bp['boxes'], period_order)):
            patch.set_facecolor(COVID_PERIODS[pid]['color'])
            patch.set_alpha(0.6)

    ax.set_title(f'{model_name}', fontweight='bold', fontsize=11)
    ax.set_ylabel('|Percentage Error| (%)' if ax == axes[0] else '')
    ax.tick_params(axis='x', rotation=45)

fig.suptitle('Error Distribution per COVID Period — Model Comparison',
             fontsize=14, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig(BENCH_DIR / 'fig_covid_error_boxplots.png', bbox_inches='tight')
plt.show()

#### 10.5.5 Detailed analysis: Holy Week under lockdown (worst case)

Zoom on period P3 (Holy Week 2020 under full lockdown): actual demand and predictions (top) and hourly percentage error of each model, with a ±5% reference band (bottom).

In [ ]:
# ============================================================
# 10.5.5  ZOOM: HOLY WEEK 2020 (P3) — EXTREME CASE
# ============================================================
P3_ID = 'P3_HolyWeek'

if P3_ID in covid_results:
    results_p3 = covid_results[P3_ID]
    y_true_p3 = results_p3['y_true']

    fig, axes = plt.subplots(2, 1, figsize=(16, 10), gridspec_kw={'height_ratios': [3, 1]})

    # Top panel: time series
    ax = axes[0]
    ax.plot(y_true_p3.index, y_true_p3.values, color='black', linewidth=2,
            label='Actual Demand', zorder=10)

    for model_name in covid_models.keys():
        if model_name in results_p3:
            preds = results_p3[model_name]['preds']
            mape = results_p3[model_name]['metrics']['MAPE']
            color = model_colors[model_name]
            ax.plot(y_true_p3.index, preds, color=color, linewidth=1.2,
                    alpha=0.8, label=f"{model_name} (MAPE: {mape:.2f}%)", linestyle='--')

    ax.set_ylabel('Demand (kWh)', fontsize=12)
    ax.set_title('Holy Week 2020 under Full Lockdown — Extreme OOD Case',
                 fontsize=14, fontweight='bold')
    ax.legend(loc='upper right', fontsize=9)

    # Annotate the days
    for day_offset in range(7):
        day = pd.Timestamp('2020-04-06') + pd.Timedelta(days=day_offset)
        day_names = ['Holy Monday', 'Holy Tuesday', 'Holy Wednesday', 'Maundy Thursday',
                     'Good Friday', 'Holy Saturday', 'Easter Sunday']
        if day in y_true_p3.index or day <= y_true_p3.index.max():
            ax.axvline(x=day, color='gray', alpha=0.3, linestyle=':')
            ax.text(day, ax.get_ylim()[0], day_names[day_offset],
                    rotation=45, fontsize=7, ha='left', va='bottom')

    # Bottom panel: hourly percentage error
    ax = axes[1]
    for model_name in covid_models.keys():
        if model_name in results_p3:
            preds = results_p3[model_name]['preds']
            valid = ~np.isnan(preds)
            error_pct = ((preds[valid] - y_true_p3.values[valid]) /
                         y_true_p3.values[valid]) * 100
            idx_valid = y_true_p3.index[valid]
            ax.plot(idx_valid, error_pct, color=model_colors[model_name],
                    alpha=0.7, linewidth=1)

    ax.axhline(y=0, color='black', linewidth=1)
    ax.axhline(y=5, color='red', linewidth=0.5, linestyle='--', alpha=0.5)
    ax.axhline(y=-5, color='red', linewidth=0.5, linestyle='--', alpha=0.5)
    ax.set_ylabel('Error (%)')
    ax.set_xlabel('Time')
    ax.fill_between(ax.get_xlim(), -5, 5, alpha=0.05, color='green')

    plt.tight_layout()
    plt.savefig(BENCH_DIR / 'fig_covid_holy_week_zoom.png', bbox_inches='tight')
    plt.show()

    # Summary table
    print("\nHourly performance, Holy Week 2020:")
    print(f"  Actual demand — Mean: {y_true_p3.mean():,.0f} | Min: {y_true_p3.min():,.0f} | "
          f"Max: {y_true_p3.max():,.0f}")
    for model_name in covid_models.keys():
        if model_name in results_p3:
            m = results_p3[model_name]['metrics']
            print(f"  {model_name:30s} — MAPE: {m['MAPE']:.2f}% | "
                  f"RMSE: {m['RMSE']:,.0f} | R²: {m['R2']:.4f}")
else:
    print("⚠️ Period P3 not available in the data")

#### 10.5.6 Conclusions of the COVID stress test

Best model per period, overall robustness (mean ± standard deviation of the MAPE across periods), hardest period (highest MAPE averaged over the models) and the difference between the mean MAPE of the MoE and that of the other models. The explanatory sentences printed after this difference are fixed texts selected by its sign, not computed results. The summary is saved to `covid_stress_conclusions.json`.

In [ ]:
# ============================================================
# 10.5.6  SUMMARY AND CONCLUSIONS OF THE STRESS TEST
# ============================================================
print("\n" + "=" * 100)
print("  CONCLUSIONS — COVID-19 STRESS TEST (out-of-distribution)")
print("=" * 100)

if len(covid_summary_df) > 0:
    # 1. Best model per period
    print("\n  1. BEST MODEL PER PERIOD:")
    for pid in period_order:
        sub = covid_summary_df[covid_summary_df['Period'] == pid]
        if len(sub) > 0:
            best = sub.loc[sub['MAPE (%)'].idxmin()]
            print(f"     {pid:20s} → {best['Model']:30s} (MAPE: {best['MAPE (%)']:.2f}%)")

    # 2. Most robust model (lowest mean MAPE across periods)
    print("\n  2. OVERALL ROBUSTNESS (mean MAPE across the COVID periods):")
    avg_by_model = covid_summary_df.groupby('Model')['MAPE (%)'].agg(['mean', 'std'])
    avg_by_model = avg_by_model.sort_values('mean')
    for name, row in avg_by_model.iterrows():
        print(f"     {name:30s} — mean MAPE: {row['mean']:.2f}% ± {row['std']:.2f}%")

    # 3. Hardest period
    print("\n  3. HARDEST PERIOD (highest mean MAPE across models):")
    avg_by_period = covid_summary_df.groupby('Period')['MAPE (%)'].mean().sort_values(ascending=False)
    for pid, mape_avg in avg_by_period.items():
        print(f"     {pid:20s} — mean MAPE: {mape_avg:.2f}%  "
              f"({COVID_PERIODS[pid]['description'][:50]})")

    # 4. MoE advantage
    print("\n  4. THE MoE ARCHITECTURE UNDER STRESS:")
    moe_rows = covid_summary_df[covid_summary_df['Family'] == 'MoE']
    if len(moe_rows) > 0:
        moe_avg = moe_rows['MAPE (%)'].mean()
        others_avg = covid_summary_df[covid_summary_df['Family'] != 'MoE']['MAPE (%)'].mean()
        advantage = others_avg - moe_avg
        print(f"     MoE mean MAPE:      {moe_avg:.2f}%")
        print(f"     Others mean MAPE:   {others_avg:.2f}%")
        if advantage > 0:
            print(f"     MoE advantage:      {advantage:.2f} percentage points")
            print(f"     Adaptive gating allows a smoother degradation out of distribution,")
            print(f"     because soft gating spreads the prediction over several experts")
            print(f"     when the classifier confidence is low.")
        else:
            print(f"     Difference:         {advantage:.2f} pp (conventional models better)")
            print(f"     On extreme OOD data, simpler models can be more robust")
            print(f"     because they do not depend on a cluster classifier.")

    # Save the summary
    covid_conclusions = {
        'best_per_period': {pid: covid_summary_df[covid_summary_df['Period'] == pid]
                            .nsmallest(1, 'MAPE (%)')['Model'].iloc[0]
                            for pid in period_order
                            if pid in covid_summary_df['Period'].values},
        'most_robust': avg_by_model.index[0],
        'hardest_period': avg_by_period.index[0],
        'avg_mape_by_model': avg_by_model['mean'].to_dict(),
    }
    with open(BENCH_DIR / 'covid_stress_conclusions.json', 'w') as f:
        json.dump(covid_conclusions, f, indent=2, default=str)

print("\n" + "=" * 100)

---
## 11. Out-of-sample forecast for January 2026

A forecasting exercise beyond the end of the dataset: the MoE and the three tuned baselines (Lasso, AdaBoost and Transformer) forecast every hour of January 2026 (744 hours), and the forecasts are compared with the actual SIN demand downloaded from the XM S.A. API.

The forecasts are **recursive**: the features of each hour are built from the calendar (with Colombian holidays from the `holidays` package) and from the demand lags. Lags that fall inside the historical data use actual demand; lags that fall inside January 2026 use the forecasts already produced by the same model. Hence errors can accumulate over the horizon. Error metrics are reported for the first week, the first two weeks and the whole month.

### 11.1 Forecast horizon and feature-generation function

`generate_features_for_timestamp` rebuilds, for one future hour, the calendar features (cyclical hour/day/month encodings, holiday and bridge-holiday flags, days to the next and from the previous holiday) and the demand lags used by the models.

In [ ]:
import holidays  # Colombian public holidays

print("Generating forecasts for January 2026...")

# 1. Forecast horizon (January 2026)
start_date_pred = pd.Timestamp('2026-01-01 00:00:00')
end_date_pred = pd.Timestamp('2026-01-31 23:00:00')
future_index = pd.date_range(start=start_date_pred, end=end_date_pred, freq='H')

# Colombian holidays over a wide range of years: earlier years are needed for
# 'days_from_prev_holiday' and later ones for 'days_to_next_holiday'
co_holidays = holidays.CO(years=range(2020, 2027))

# DataFrame that stores the forecasts of all models
df_pred_all_models = pd.DataFrame(index=future_index)

In [ ]:
def generate_features_for_timestamp(ts, historical_df, future_df, feature_cols, target_col, co_holidays):
    """Build the features of one timestamp.
    historical_df: DataFrame with the actual historical data (up to the end of the dataset)
    future_df: DataFrame with the forecasts already generated for the future (used for the lags)
    """
    features = {}

    # 1. Cyclical time features
    features['hour_sin'] = np.sin(2 * np.pi * ts.hour / 24)
    features['hour_cos'] = np.cos(2 * np.pi * ts.hour / 24)
    features['dayofweek_sin'] = np.sin(2 * np.pi * ts.dayofweek / 7)
    features['dayofweek_cos'] = np.cos(2 * np.pi * ts.dayofweek / 7)
    features['month_sin'] = np.sin(2 * np.pi * ts.month / 12)
    features['month_cos'] = np.cos(2 * np.pi * ts.month / 12)

    # 2. Holiday features
    features['is_holiday'] = 1 if ts.date() in co_holidays else 0

    features['is_bridge'] = 0
    if ts.weekday() == 0 and (ts.date() - timedelta(days=3)) in co_holidays:
        features['is_bridge'] = 1
    if ts.weekday() == 4 and (ts.date() + timedelta(days=3)) in co_holidays:
        features['is_bridge'] = 1
    if (ts.date() - timedelta(days=1)) in co_holidays and (ts.date() + timedelta(days=1)) in co_holidays:
        features['is_bridge'] = 1

    features['days_to_next_holiday'] = 0
    features['days_from_prev_holiday'] = 0

    if ts.date() not in co_holidays:
        next_holiday_date = None
        for day_offset in range(1, 366):
            future_date = ts.date() + timedelta(days=day_offset)
            if future_date in co_holidays:
                next_holiday_date = future_date
                break
        if next_holiday_date:
            features['days_to_next_holiday'] = (next_holiday_date - ts.date()).days
        else:
            features['days_to_next_holiday'] = 365

        prev_holiday_date = None
        for day_offset in range(1, 366):
            past_date = ts.date() - timedelta(days=day_offset)
            if past_date in co_holidays:
                prev_holiday_date = past_date
                break
        if prev_holiday_date:
            features['days_from_prev_holiday'] = (ts.date() - prev_holiday_date).days
        else:
            features['days_from_prev_holiday'] = 365

    # 3. Demand lags
    # Make sure the indexes are datetime before concatenating and looking up
    if not isinstance(historical_df.index, pd.DatetimeIndex):
        historical_df.index = pd.to_datetime(historical_df.index)
    if not isinstance(future_df.index, pd.DatetimeIndex):
        future_df.index = pd.to_datetime(future_df.index)

    # Combine historical and future data for the lag lookup.
    # historical_df[target_col] provides only actual demand values;
    # future_df[target_col].dropna() keeps only the forecasts already made
    combined_data = pd.concat([historical_df[target_col], future_df[target_col].dropna()]).sort_index()

    lag_cols_nums = sorted([int(c.split('_')[-1]) for c in feature_cols if c.startswith('demand_lag_')])
    for lag in lag_cols_nums:
        lag_ts = ts - timedelta(hours=lag)
        if lag_ts in combined_data.index:
            features[f'demand_lag_{lag}'] = combined_data.loc[lag_ts]
        else:
            # If the lag is not available (e.g., ts is too early in the history), use NaN.
            # Models must then handle NaN, or the input must be preprocessed.
            # Here df_clean is expected to cover the required range.
            features[f'demand_lag_{lag}'] = np.nan

    # Keep only the features listed in feature_cols
    return {k: v for k, v in features.items() if k in feature_cols}

The next cell defines the functions that download the actual hourly SIN demand (metric `DemaReal`, entity `Sistema`) from the XM S.A. API, one day per request, with retries and exponential back-off. They are a copy of the download logic of notebook 01 and are used in Section 11.6.

In [ ]:
from typing import List, Tuple, Optional, Dict, Any
import requests

# ============================================================
# Ingestion functions — XM API (servapibi.xm.com.co)
# Windows and payload aligned with the reference JavaScript client (ReadDB)
# ============================================================
API_URL = "https://servapibi.xm.com.co/hourly"
DEMAND_VARIABLE = "DemaReal"
MAX_RETRIES = 3
INITIAL_BACKOFF = 2  # seconds

# Valid historical range of hourly SIN demand (kWh): ~4,000-12,000 MWh
DEMAND_MIN_KWH = 4_000_000
DEMAND_MAX_KWH = 12_000_000


def generate_date_windows(
    start_date: str, end_date: str
) -> List[Tuple[str, str]]:
    """
    Generate the date windows used to iterate over the XM API.

    Hourly data (HourlyEntities) are requested in ONE-DAY windows, as in the
    reference JavaScript client: 24 records per request. Longer windows (e.g.
    30 days = 720 records) can exceed the internal limit of the API and return
    empty or truncated responses.

    Parameters
    ----------
    start_date : str  — Start date 'YYYY-MM-DD'.
    end_date : str    — End date 'YYYY-MM-DD'.

    Returns
    -------
    List[Tuple[str, str]]
        List of (start, end) tuples with start == end (one day per window).
    """
    windows = []
    current = datetime.strptime(start_date, "%Y-%m-%d")
    end = datetime.strptime(end_date, "%Y-%m-%d")

    while current <= end:
        # One-day window: start == end (as in the JavaScript client)
        date_str = current.strftime("%Y-%m-%d")
        windows.append((date_str, date_str))
        current += timedelta(days=1)

    return windows


def _parse_hourly_items(items: list) -> list:
    """
    Parse the XM API response, supporting BOTH possible formats of
    HourlyEntities:

    Format A (list of {Hour, Value}):
        "HourlyEntities": [
            {"Hour": 1, "Value": 5432.1},
            {"Hour": 2, "Value": 5210.3}, ...
        ]

    Format B (list of entities with a Values dict):
        "HourlyEntities": [
            {
                "Id": "Sistema",
                "Values": {"Hour01": 5432.1, "Hour02": 5210.3, ...}
            }
        ]

    This defensive parsing avoids silent failures if the structure of the
    response does not match the expected one.
    """
    records = []

    for item in items:
        date_str = item.get("Date", "")
        if not date_str:
            continue

        base_date = pd.to_datetime(date_str).normalize()
        hourly_entities = item.get("HourlyEntities", [])

        for entity in hourly_entities:
            # --- Format A: each entity has Hour and Value directly ---
            if "Hour" in entity and "Value" in entity:
                hour = entity["Hour"]  # hours 1-24 in the API
                value = entity.get("Value")
                if value is not None:
                    dt = base_date + pd.Timedelta(hours=int(hour) - 1)
                    records.append({"datetime": dt, "demand_kwh": float(value)})

            # --- Format B: entity has Id + Values dict ---
            elif "Values" in entity:
                values_dict = entity["Values"]
                for key, value in values_dict.items():
                    # Keys such as "Hour01", "Hour02", ..., "Hour24"
                    if key.startswith("Hour") and value is not None:
                        try:
                            hour = int(key.replace("Hour", ""))
                            dt = base_date + pd.Timedelta(hours=hour - 1)
                            records.append({"datetime": dt, "demand_kwh": float(value)})
                        except (ValueError, TypeError):
                            continue

            # --- Unknown format: log it for debugging ---
            else:
                print(f"  [WARN] Unexpected structure in HourlyEntities: {list(entity.keys())}")

    return records


def fetch_xm_demand(
    start_date: str,
    end_date: str,
    max_retries: int = MAX_RETRIES,
    initial_backoff: float = INITIAL_BACKOFF,
) -> Optional[pd.DataFrame]:
    """
    Download the actual hourly SIN demand from the XM API for one date
    window (ideally 1 day).

    Notes:
    1. The payload includes "Filter": [], as in the JavaScript client. Some
       APIs check that every schema field is present, even if empty.
    2. Defensive parsing that supports several response formats.

    Parameters
    ----------
    start_date : str  — Start date 'YYYY-MM-DD'.
    end_date : str    — End date 'YYYY-MM-DD'.
    max_retries : int — Maximum number of retries.
    initial_backoff : float — Initial back-off in seconds.

    Returns
    -------
    pd.DataFrame or None
    """
    payload = {
        "MetricId": DEMAND_VARIABLE,
        "StartDate": start_date,
        "EndDate": end_date,
        "Entity": "Sistema",
        "Filter": [],  # ← field present in the JavaScript client, missing in Python
    }
    headers = {"Content-Type": "application/json"}

    for attempt in range(1, max_retries + 1):
        try:
            response = requests.post(
                API_URL, json=payload, headers=headers, timeout=60
            )
            response.raise_for_status()
            data = response.json()

            # Defensive parsing that supports both response formats
            items = data.get("Items", [])
            records = _parse_hourly_items(items)

            if records:
                return pd.DataFrame(records)
            else:
                print(f"  [WARN] No records for {start_date} -> {end_date}")
                return None

        except requests.exceptions.RequestException as e:
            wait = initial_backoff * (2 ** (attempt - 1))
            print(
                f"  [RETRY {attempt}/{max_retries}] Error: {e}. "
                f"Retrying in {wait:.0f}s..."
            )
            time.sleep(wait)

    print(f"  [ERROR] All attempts failed for {start_date} -> {end_date}")
    return None


def download_demand_data(
    start_date: str,
    end_date: str,
) -> pd.DataFrame:
    """
    Download the hourly SIN demand for an arbitrary date range, iterating
    day by day (as in the JavaScript client).

    Efficiency note: with 1-day windows, one year requires ~365 requests.
    With a 0.3 s pause per request this takes ~2 min. This is the right
    trade-off: the API answers consistently, whereas large windows come back
    empty.

    Parameters
    ----------
    start_date : str  — Start date 'YYYY-MM-DD'.
    end_date : str    — End date 'YYYY-MM-DD'.

    Returns
    -------
    pd.DataFrame
        Consolidated DataFrame ['datetime', 'demand_kwh'], sorted, without duplicates.
    """
    windows = generate_date_windows(start_date, end_date)
    print(f"Total windows (days) to request: {len(windows)}")
    print(f"Period: {start_date} -> {end_date}")
    print("=" * 60)

    all_dfs = []
    failed_days = []

    for i, (ws, we) in enumerate(windows, 1):
        df = fetch_xm_demand(ws, we)
        if df is not None:
            all_dfs.append(df)
        else:
            failed_days.append(ws)
            print("NO DATA")

        # Pause between requests so as not to overload the API
        # (the JavaScript client uses Promise.all without throttling; a conservative pace is safer)
        time.sleep(0.3)

    if not all_dfs:
        raise RuntimeError("No data were obtained from the API.")

    # Consolidate
    df_full = pd.concat(all_dfs, ignore_index=True)

    # Remove duplicated timestamps
    n_before = len(df_full)
    df_full = df_full.drop_duplicates(subset="datetime", keep="first")
    n_dupes = n_before - len(df_full)
    if n_dupes > 0:
        print(f"\n[INFO] {n_dupes} duplicated timestamps were removed.")

    # Sort
    df_full = df_full.sort_values("datetime").reset_index(drop=True)

    print(f"\n{'=' * 60}")
    print(f"Download completed: {len(df_full):,} records")
    print(f"Range: {df_full['datetime'].min()} -> {df_full['datetime'].max()}")

    if failed_days:
        print(f"[WARN] Days without data ({len(failed_days)}): {failed_days[:10]}...")

    return df_full

### 11.2 MoE forecast

Recursive hourly forecast of January 2026 with the production MoE.

In [ ]:
# DataFrame that stores the future features and forecasts
df_pred_features = pd.DataFrame(index=future_index, columns=calendar_lags + [TARGET])

if not isinstance(df_clean.index, pd.DatetimeIndex):
    df_clean.index = pd.to_datetime(df_clean.index)

for i, ts in enumerate(future_index):
    # Build the features of the current timestamp
    current_features_dict = generate_features_for_timestamp(ts, df_clean, df_pred_features, calendar_lags, TARGET, co_holidays)

    # One-row DataFrame for the prediction
    X_current = pd.DataFrame([current_features_dict], index=[ts], columns=calendar_lags)

    # MoE prediction
    pred = moe_model.predict(X_current)[0]
    df_pred_features.loc[ts, TARGET] = pred

    if (i + 1) % 100 == 0 or i == len(future_index) - 1:
        print(f"  Forecast {i + 1}/{len(future_index)} hours of January 2026. Last forecast: {ts} -> {pred:,.0f} kWh")

print("\nJanuary 2026 forecast completed for MoE.")

df_pred_all_models['pred_moe'] = df_pred_features[TARGET]

# Plot the forecast
plt.figure(figsize=(15, 6))
plt.plot(df_pred_features.index, df_pred_features[TARGET], label='Demand Forecast (kWh)', color='blue')
plt.title('Electricity Demand Forecast for January 2026 (MoE)')
plt.xlabel('Date')
plt.ylabel('Demand (kWh)')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

### 11.3 Lasso forecast

Recursive forecast with the tuned Lasso model (first entry of `hpo_results`).

In [ ]:
model_lasso = list(hpo_results.items())[0][1]['model']

In [ ]:
# DataFrame that stores the future features and forecasts
df_pred_features = pd.DataFrame(index=future_index, columns=calendar_lags + [TARGET])

if not isinstance(df_clean.index, pd.DatetimeIndex):
    df_clean.index = pd.to_datetime(df_clean.index)

for i, ts in enumerate(future_index):
    # Build the features of the current timestamp
    current_features_dict = generate_features_for_timestamp(ts, df_clean, df_pred_features, calendar_lags, TARGET, co_holidays)

    # One-row DataFrame for the prediction
    X_current = pd.DataFrame([current_features_dict], index=[ts], columns=calendar_lags)

    # Lasso prediction
    pred = model_lasso.predict(X_current)[0]
    df_pred_features.loc[ts, TARGET] = pred

    if (i + 1) % 100 == 0 or i == len(future_index) - 1:
        print(f"  Forecast {i + 1}/{len(future_index)} hours of January 2026. Last forecast: {ts} -> {pred:,.0f} kWh")

print("\nJanuary 2026 forecast completed for Lasso.")

df_pred_all_models['pred_lasso'] = df_pred_features[TARGET]

# Plot the forecast
plt.figure(figsize=(15, 6))
plt.plot(df_pred_features.index, df_pred_features[TARGET], label='Demand Forecast (kWh)', color='blue')
plt.title('Electricity Demand Forecast for January 2026 (Lasso)')
plt.xlabel('Date')
plt.ylabel('Demand (kWh)')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

### 11.4 AdaBoost forecast

Recursive forecast with the tuned AdaBoost model (second entry of `hpo_results`).

In [ ]:
model_adaboost = list(hpo_results.items())[1][1]['model']

In [ ]:
# DataFrame that stores the future features and forecasts
df_pred_features = pd.DataFrame(index=future_index, columns=calendar_lags + [TARGET])

if not isinstance(df_clean.index, pd.DatetimeIndex):
    df_clean.index = pd.to_datetime(df_clean.index)

for i, ts in enumerate(future_index):
    # Build the features of the current timestamp
    current_features_dict = generate_features_for_timestamp(ts, df_clean, df_pred_features, calendar_lags, TARGET, co_holidays)

    # One-row DataFrame for the prediction
    X_current = pd.DataFrame([current_features_dict], index=[ts], columns=calendar_lags)

    # Adaboost prediction
    pred = model_adaboost.predict(X_current)[0]
    df_pred_features.loc[ts, TARGET] = pred

    if (i + 1) % 100 == 0 or i == len(future_index) - 1:
        print(f"  Forecast {i + 1}/{len(future_index)} hours of January 2026. Last forecast: {ts} -> {pred:,.0f} kWh")

print("\nJanuary 2026 forecast completed for Adaboost.")

df_pred_all_models['pred_adaboost'] = df_pred_features[TARGET]

# Plot the forecast
plt.figure(figsize=(15, 6))
plt.plot(df_pred_features.index, df_pred_features[TARGET], label='Demand Forecast (kWh)', color='blue')
plt.title('Electricity Demand Forecast for January 2026 (Adaboost)')
plt.xlabel('Date')
plt.ylabel('Demand (kWh)')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

### 11.5 Transformer forecast

Recursive forecast with the tuned Transformer (first entry of `dl_hpo_results`). Each prediction needs an input window of 168 hours. At the start of the month the window is filled with the last rows of the historical feature table followed by the rows of the hours already forecast; the window is standardized with `scaler_X` and the output is transformed back with `scaler_y`.

In [ ]:
model_transformers = list(dl_hpo_results.items())[0][1]['model']

In [ ]:
df_pred_features = pd.DataFrame(index=future_index)

if not isinstance(df_clean.index, pd.DatetimeIndex):
    df_clean.index = pd.to_datetime(df_clean.index)

model_transformers.eval()

# DataFrame that stores the future features and forecasts
df_pred_features = pd.DataFrame(index=future_index, columns=calendar_lags + [TARGET])

for i, ts in enumerate(future_index):
    # Build the features of the current timestamp
    current_features_dict = generate_features_for_timestamp(
        ts, df_clean, df_pred_features, calendar_lags, TARGET, co_holidays
    )

    # One-row DataFrame
    X_current = pd.DataFrame([current_features_dict], index=[ts], columns=calendar_lags)

    # Scale the current row with the input scaler
    X_current_sc = scaler_X.transform(X_current)

    # ---------------------------------------------------------
    # Build the input sequence of length WINDOW_SIZE
    # from actual history + rows of the hours already forecast
    # ---------------------------------------------------------
    if i == 0:
        hist_input = df_clean[calendar_lags].iloc[-(WINDOW_SIZE-1):].copy()
        X_seq = pd.concat([hist_input, X_current], axis=0)
    else:
        prev_rows = df_pred_features[calendar_lags].iloc[:i].copy()

        hist_needed = WINDOW_SIZE - len(prev_rows) - 1

        if hist_needed > 0:
            hist_input = df_clean[calendar_lags].iloc[-hist_needed:].copy()
            X_seq = pd.concat([hist_input, prev_rows, X_current], axis=0)
        else:
            X_seq = pd.concat([prev_rows.iloc[-(WINDOW_SIZE-1):], X_current], axis=0)

    # Enforce the exact column order
    X_seq = X_seq[calendar_lags]

    # Scale the whole sequence
    X_seq_sc = scaler_X.transform(X_seq)

    # Tensor of shape (1, WINDOW_SIZE, n_features)
    X_tensor = torch.tensor(X_seq_sc, dtype=torch.float32).unsqueeze(0).to(device)

    # Deep-learning prediction
    with torch.no_grad():
        pred_sc = model_transformers(X_tensor).cpu().numpy().reshape(-1, 1)

    pred = scaler_y.inverse_transform(pred_sc).flatten()[0]

    # Store the forecast
    df_pred_features.loc[ts, TARGET] = pred

    if (i + 1) % 100 == 0 or i == len(future_index) - 1:
        print(f"  Forecast {i + 1}/{len(future_index)} hours of January 2026. Last forecast: {ts} -> {pred:,.0f} kWh")


print(f"\nJanuary 2026 forecast completed for Transformers.")

df_pred_all_models['pred_transformers'] = df_pred_features[TARGET]

# Plot the forecast
plt.figure(figsize=(15, 6))
plt.plot(df_pred_features.index, df_pred_features[TARGET], label='Demand Forecast (kWh)', color='blue')
plt.title('Electricity Demand Forecast for January 2026 (Transformers)')
plt.xlabel('Date')
plt.ylabel('Demand (kWh)')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

### 11.6 Download of the actual demand from XM S.A.

Actual hourly SIN demand for January 2026 (requires internet access).

In [ ]:
# Download the data
START_DATE = "2026-01-01"
END_DATE = "2026-01-31"

df_demand_real_january = download_demand_data(START_DATE, END_DATE)

In [ ]:
# Use the datetime column as a sorted index, aligned with the forecasts
df_demand_real_january = df_demand_real_january.set_index('datetime').sort_index()

In [ ]:
df_demand_real_january

### 11.7 Comparison with the actual demand

Table of the hourly forecasts of the four models (`pred_moe`, `pred_lasso`, `pred_adaboost`, `pred_transformers`).

In [ ]:
df_pred_all_models

#### 11.7.1 Figures

Forecasts vs. actual demand for the whole month, the first week (Jan 1–7) and the first two weeks (Jan 1–14). The figures are saved to `BENCH_DIR`.

In [ ]:
plt.figure(figsize=(15, 8))

# Forecasts
for col in df_pred_all_models.columns:
    plt.plot(df_pred_all_models.index, df_pred_all_models[col], label=f"Prediction {col.replace('pred_', '', 1).capitalize().replace('Moe', 'MoE')}", alpha=0.7)

# Actual demand, if available
if 'df_demand_real_january' in locals() and not df_demand_real_january.empty:
    plt.plot(df_demand_real_january.index, df_demand_real_january['demand_kwh'], label='Actual Demand (XM)', color='black', linewidth=2, alpha=0.8)

plt.title('Electricity Demand Forecast for January 2026', fontsize=16)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Demand (kWh)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=10, loc='upper left')
plt.tight_layout()
plt.savefig(BENCH_DIR / 'demand_forecast_january_2026.png')
plt.show()

In [ ]:
plt.figure(figsize=(15, 5))

# First week of January 2026
start_date_week1 = pd.Timestamp('2026-01-01 00:00:00')
end_date_week1 = pd.Timestamp('2026-01-07 23:00:00')

# Forecasts in the window
df_pred_week1 = df_pred_all_models.loc[start_date_week1:end_date_week1]

# Actual demand in the window
df_demand_real_week1 = df_demand_real_january.loc[start_date_week1:end_date_week1]

# Forecasts
for col in df_pred_week1.columns:
    plt.plot(df_pred_week1.index, df_pred_week1[col], label=f"Prediction {col.replace('pred_', '', 1).capitalize().replace('Moe', 'MoE')}", alpha=0.7)

# Actual demand, if available
if not df_demand_real_week1.empty:
    plt.plot(df_demand_real_week1.index, df_demand_real_week1['demand_kwh'], label='Actual Demand (XM)', color='black', linewidth=2, alpha=0.8)

plt.title('Electricity Demand Forecast for the First Week of January 2026 — Model Comparison', fontsize=16)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Demand (kWh)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=10, loc='upper left')
plt.tight_layout()
plt.savefig(BENCH_DIR / 'demand_forecast_january_2026_one_week.png')
plt.show()

In [ ]:
plt.figure(figsize=(15, 5))

# First two weeks of January 2026
start_date_week1 = pd.Timestamp('2026-01-01 00:00:00')
end_date_week1 = pd.Timestamp('2026-01-14 23:00:00')

# Forecasts in the window
df_pred_week1 = df_pred_all_models.loc[start_date_week1:end_date_week1]

# Actual demand in the window
df_demand_real_week1 = df_demand_real_january.loc[start_date_week1:end_date_week1]

# Forecasts
for col in df_pred_week1.columns:
    plt.plot(df_pred_week1.index, df_pred_week1[col], label=f"Prediction {col.replace('pred_', '', 1).capitalize().replace('Moe', 'MoE')}", alpha=0.7)

# Actual demand, if available
if not df_demand_real_week1.empty:
    plt.plot(df_demand_real_week1.index, df_demand_real_week1['demand_kwh'], label='Actual Demand (XM)', color='black', linewidth=2, alpha=0.8)

plt.title('Electricity Demand Forecast for the First 2 Weeks of January 2026 — Model Comparison', fontsize=16)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Demand (kWh)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=10, loc='upper left')
plt.tight_layout()
plt.savefig(BENCH_DIR / 'demand_forecast_january_2026_two_week.png')
plt.show()

#### 11.7.2 Error metrics

MAE, RMSE, MAPE and R² of each model over the hours where both the forecast and the actual demand are available, for the whole month (`metrics_jan_2026.csv`), the first week (`metrics_jan_2026_one_week.csv`) and the first two weeks (`metrics_jan_2026_two_week.csv`).

In [ ]:
metrics_jan_2026 = []

# Align the DataFrames by index before computing the metrics
common_index = df_pred_all_models.index.intersection(df_demand_real_january.index)

y_true_jan = df_demand_real_january.loc[common_index, 'demand_kwh']

for col in df_pred_all_models.columns:
    model_name = col.replace('pred_', '')
    y_pred_jan = df_pred_all_models.loc[common_index, col].fillna(0)

    valid_indices = y_pred_jan.dropna().index
    if len(valid_indices) > 0:
        metrics = compute_metrics(y_true_jan.loc[valid_indices], y_pred_jan.loc[valid_indices])
        metrics_jan_2026.append({
            'Model': model_name,
            'MAE': metrics['MAE'],
            'RMSE': metrics['RMSE'],
            'MAPE': metrics['MAPE'],
            'R2': metrics['R2']
        })
    else:
        metrics_jan_2026.append({
            'Model': model_name,
            'MAE': np.nan,
            'RMSE': np.nan,
            'MAPE': np.nan,
            'R2': np.nan
        })

df_metrics_jan_2026 = pd.DataFrame(metrics_jan_2026).sort_values('MAPE')

print("\n" + "="*80)
print("  ERROR METRICS FOR JANUARY 2026 — MODEL COMPARISON")
print("="*80)
print(df_metrics_jan_2026.to_string(index=False, float_format='{:.4f}'.format))

df_metrics_jan_2026.to_csv(BENCH_DIR / 'metrics_jan_2026.csv', index=False)

In [ ]:
metrics_jan_2026 = []

# Window: first week of January 2026
start_date_week1 = pd.Timestamp('2026-01-01 00:00:00')
end_date_week1 = pd.Timestamp('2026-01-07 23:00:00')

# Forecasts in the window
df_pred_week1 = df_pred_all_models.loc[start_date_week1:end_date_week1]

# Actual demand in the window
df_demand_real_week1 = df_demand_real_january.loc[start_date_week1:end_date_week1]

# Align the DataFrames by index before computing the metrics
common_index = df_pred_week1.index.intersection(df_demand_real_week1.index)

y_true_jan = df_demand_real_week1.loc[common_index, 'demand_kwh']

for col in df_pred_week1.columns:
    model_name = col.replace('pred_', '')
    y_pred_jan = df_pred_week1.loc[common_index, col].fillna(0)

    valid_indices = y_pred_jan.dropna().index
    if len(valid_indices) > 0:
        metrics = compute_metrics(y_true_jan.loc[valid_indices], y_pred_jan.loc[valid_indices])
        metrics_jan_2026.append({
            'Model': model_name,
            'MAE': metrics['MAE'],
            'RMSE': metrics['RMSE'],
            'MAPE': metrics['MAPE'],
            'R2': metrics['R2']
        })
    else:
        metrics_jan_2026.append({
            'Model': model_name,
            'MAE': np.nan,
            'RMSE': np.nan,
            'MAPE': np.nan,
            'R2': np.nan
        })

df_metrics_jan_2026 = pd.DataFrame(metrics_jan_2026).sort_values('MAPE')

print("\n" + "="*80)
print("  ERROR METRICS FOR JANUARY 2026 — MODEL COMPARISON (first week)")
print("="*80)
print(df_metrics_jan_2026.to_string(index=False, float_format='{:.4f}'.format))


df_metrics_jan_2026.to_csv(BENCH_DIR / 'metrics_jan_2026_one_week.csv', index=False)

In [ ]:
metrics_jan_2026 = []

# Window: first two weeks of January 2026
start_date_week1 = pd.Timestamp('2026-01-01 00:00:00')
end_date_week1 = pd.Timestamp('2026-01-14 23:00:00')

# Forecasts in the window
df_pred_week1 = df_pred_all_models.loc[start_date_week1:end_date_week1]

# Actual demand in the window
df_demand_real_week1 = df_demand_real_january.loc[start_date_week1:end_date_week1]

# Align the DataFrames by index before computing the metrics
common_index = df_pred_week1.index.intersection(df_demand_real_week1.index)

y_true_jan = df_demand_real_week1.loc[common_index, 'demand_kwh']

for col in df_pred_week1.columns:
    model_name = col.replace('pred_', '')
    y_pred_jan = df_pred_week1.loc[common_index, col].fillna(0)

    valid_indices = y_pred_jan.dropna().index
    if len(valid_indices) > 0:
        metrics = compute_metrics(y_true_jan.loc[valid_indices], y_pred_jan.loc[valid_indices])
        metrics_jan_2026.append({
            'Model': model_name,
            'MAE': metrics['MAE'],
            'RMSE': metrics['RMSE'],
            'MAPE': metrics['MAPE'],
            'R2': metrics['R2']
        })
    else:
        metrics_jan_2026.append({
            'Model': model_name,
            'MAE': np.nan,
            'RMSE': np.nan,
            'MAPE': np.nan,
            'R2': np.nan
        })

df_metrics_jan_2026 = pd.DataFrame(metrics_jan_2026).sort_values('MAPE')

print("\n" + "="*80)
print("  ERROR METRICS FOR JANUARY 2026 — MODEL COMPARISON (first two weeks)")
print("="*80)
print(df_metrics_jan_2026.to_string(index=False, float_format='{:.4f}'.format))

df_metrics_jan_2026.to_csv(BENCH_DIR / 'metrics_jan_2026_two_week.csv', index=False)

---
## 12. Executive summary and export of results

Best model on the test set (first row of `summary_df`, ranked by test MAPE), best model of each family, p-values of the ANOVA and Friedman tests, and export of the final summary to `benchmark_results.json`. The cell ends by listing every file in `BENCH_DIR`.

In [ ]:
# ============================================================
# 12. FINAL SUMMARY
# ============================================================
print("\n" + "="*100)
print("  EXECUTIVE SUMMARY — COMPARATIVE BENCHMARK")
print("="*100)

# Best model overall
best_model = summary_df.iloc[0]
print(f"\n  🏆 BEST MODEL: {best_model['Model']}")
print(f"     Family:   {best_model['Family']}")
print(f"     MAPE:     {best_model['MAPE (%)']:.4f}%")
print(f"     RMSE:     {best_model['RMSE']:.2f}")
print(f"     MAE:      {best_model['MAE']:.2f}")
print(f"     R²:       {best_model['R²']:.4f}")

# Best per family
print("\n  Best per family:")
for family in ['MoE', 'Statistical', 'ML', 'Deep Learning']:
    fam_df = summary_df[summary_df['Family'] == family]
    if len(fam_df) > 0:
        best = fam_df.iloc[0]
        print(f"    {family:15s}: {best['Model']:25s} — MAPE: {best['MAPE (%)']:.4f}%")

# Statistical significance
print(f"\n  Statistical tests:")
print(f"    ANOVA p-value:   {p_value_anova:.2e} {'✓' if p_value_anova < 0.05 else '✗'}")
try:
    print(f"    Friedman p-value: {p_value_friedman:.2e} {'✓' if p_value_friedman < 0.05 else '✗'}")
except:
    pass

# Save everything
results_export = {
    'summary': summary_df.to_dict('records'),
    'anova_f_stat': float(f_stat),
    'anova_p_value': float(p_value_anova),
    'best_model': best_model['Model'],
    'timestamp': datetime.now().isoformat(),
    'n_random_search_iter': N_RANDOM_SEARCH_ITER,
    'methodology': 'Bergstra & Bengio (2012) Random Search',
}

with open(BENCH_DIR / 'benchmark_results.json', 'w') as f:
    json.dump(results_export, f, indent=2, default=str)

print(f"\n  Files generated in {BENCH_DIR}:")
for f in sorted(BENCH_DIR.iterdir()):
    print(f"    ✓ {f.name}")

print("\n" + "="*100)
print("  ✅ BENCHMARK COMPLETE")
print("="*100)